In [15]:
# ================================================================
# Cell 0 — TSC forward-only inference with Adaptive Group LASSO
#
# Purpose
# -------
# Reconstruct the persistent and chronology-induced effective
# vector fields from forward multi-resolution endpoint data only.
#
# Physical system:
#   N = 8 nonlinear temporal network
#   pairwise temporal interactions
#   + persistent native triadic interactions
#
# Learner observes only:
#   (x0, xF(epsilon), epsilon)
#
# Reverse trajectories are never generated.
# Oracle information is reserved for final validation only.
# ================================================================

from pathlib import Path
from itertools import combinations, combinations_with_replacement
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import sympy as sp


# --------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------

SEED = 20260811

weight_rng = np.random.default_rng(SEED)
data_rng = np.random.default_rng(SEED + 202)


# --------------------------------------------------------------
# Output directories
# --------------------------------------------------------------

DATA_DIR = Path("./data_adaptive_gl")
FIGURE_DIR = Path("./figures_adaptive_gl")

DATA_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)


# --------------------------------------------------------------
# System size
# --------------------------------------------------------------

N = 8
nodes = tuple(range(1, N + 1))

x_symbols = sp.symbols(
    f"x1:{N + 1}"
)


# --------------------------------------------------------------
# Physical nonlinear interaction
# --------------------------------------------------------------

LAMBDA = 0.5
LAMBDA_EXACT = sp.Rational(1, 2)


# --------------------------------------------------------------
# Dataset configuration
# --------------------------------------------------------------

EPS_VALUES = np.array([
    0.005,
    0.0075,
    0.010,
    0.015,
    0.020,
    0.030,
    0.040,
    0.060,
    0.080,
    0.100,
    0.120,
], dtype=float)

N_TRAIN = 3000
N_TEST = 1000
N_TOTAL = N_TRAIN + N_TEST

MAX_RK4_STEP = 1e-3


# --------------------------------------------------------------
# Generic inference library
# --------------------------------------------------------------

MAX_POLY_DEGREE = 3
MAX_SUPPORT_SIZE = 4


print("TSC Adaptive Group-LASSO notebook initialized.")
print("N =", N)
print("seed =", SEED)
print("train / test =", N_TRAIN, "/", N_TEST)
print("epsilon values =", EPS_VALUES)
print(
    "generic library:",
    f"degree <= {MAX_POLY_DEGREE},",
    f"support <= {MAX_SUPPORT_SIZE}"
)

TSC Adaptive Group-LASSO notebook initialized.
N = 8
seed = 20260811
train / test = 3000 / 1000
epsilon values = [0.005  0.0075 0.01   0.015  0.02   0.03   0.04   0.06   0.08   0.1
 0.12  ]
generic library: degree <= 3, support <= 4


Data generation

In [16]:
# ================================================================
# Cell 1 — Frozen N=8 temporal network
# ================================================================

# --------------------------------------------------------------
# Microscopic pairwise graph
# --------------------------------------------------------------

microscopic_edges = (
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),
    (6, 7),
    (7, 8),
    (8, 1),
    (1, 3),
    (2, 4),
    (3, 5),
    (5, 7),
)

assert len(microscopic_edges) == 12
assert len(set(microscopic_edges)) == 12


# --------------------------------------------------------------
# Frozen heterogeneous edge weights
#
# Same construction as the original Stage 4 benchmark:
# integer weights sampled once on [800, 1200], then divided by 1000.
# --------------------------------------------------------------

edge_weight_integer = {
    edge: int(weight_rng.integers(800, 1201))
    for edge in microscopic_edges
}

edge_weight = {
    edge: value / 1000.0
    for edge, value in edge_weight_integer.items()
}

edge_weight_exact = {
    edge: sp.Rational(value, 1000)
    for edge, value in edge_weight_integer.items()
}


# --------------------------------------------------------------
# Frozen six-snapshot temporal protocol
#
# Each microscopic edge appears exactly twice over the full
# six-snapshot protocol.
# --------------------------------------------------------------

snapshots = (
    ((1, 2), (3, 4), (5, 6), (7, 8)),
    ((2, 3), (4, 5), (6, 7), (8, 1)),
    ((1, 3), (2, 4), (3, 5), (5, 7)),
    ((1, 2), (4, 5), (7, 8), (3, 5)),
    ((2, 3), (5, 6), (8, 1), (5, 7)),
    ((3, 4), (6, 7), (1, 3), (2, 4)),
)


# --------------------------------------------------------------
# Consistency audit
# --------------------------------------------------------------

snapshot_edge_counts = Counter(
    edge
    for snapshot in snapshots
    for edge in snapshot
)

assert len(snapshots) == 6
assert all(len(snapshot) == 4 for snapshot in snapshots)

assert set(snapshot_edge_counts) == set(microscopic_edges)

assert all(
    snapshot_edge_counts[edge] == 2
    for edge in microscopic_edges
)


# --------------------------------------------------------------
# Compact summary
# --------------------------------------------------------------

print("Frozen microscopic network:")
print("  nodes =", N)
print("  edges =", len(microscopic_edges))
print("  snapshots =", len(snapshots))
print("  edges per snapshot =", len(snapshots[0]))
print("  exposure per microscopic edge = 2")

print()
print("Frozen edge weights:")
for edge in microscopic_edges:
    print(
        f"  {edge}: {edge_weight[edge]:.3f}"
    )

print()
print("Temporal protocol:")
for r, snapshot in enumerate(snapshots, start=1):
    print(f"  G{r} = {snapshot}")

Frozen microscopic network:
  nodes = 8
  edges = 12
  snapshots = 6
  edges per snapshot = 4
  exposure per microscopic edge = 2

Frozen edge weights:
  (1, 2): 1.006
  (2, 3): 0.911
  (3, 4): 1.105
  (4, 5): 0.917
  (5, 6): 1.177
  (6, 7): 1.119
  (7, 8): 0.917
  (8, 1): 0.944
  (1, 3): 1.026
  (2, 4): 1.002
  (3, 5): 1.067
  (5, 7): 1.031

Temporal protocol:
  G1 = ((1, 2), (3, 4), (5, 6), (7, 8))
  G2 = ((2, 3), (4, 5), (6, 7), (8, 1))
  G3 = ((1, 3), (2, 4), (3, 5), (5, 7))
  G4 = ((1, 2), (4, 5), (7, 8), (3, 5))
  G5 = ((2, 3), (5, 6), (8, 1), (5, 7))
  G6 = ((3, 4), (6, 7), (1, 3), (2, 4))


In [17]:
# ================================================================
# Cell 2 — Microscopic nonlinear vector fields
#
# Pairwise temporal dynamics:
#     phi(x) = x + 0.5 x^2
#
# Persistent native triads:
#     H_123 and H_258
#     g = 0.02
#
# Both symbolic and numerical/vectorized implementations are
# defined from the same frozen physical model.
# ================================================================


# --------------------------------------------------------------
# Nonlinear pairwise interaction
# --------------------------------------------------------------

def phi_symbolic(z):
    return z + LAMBDA_EXACT * z**2


def phi_numeric(z):
    return z + LAMBDA * z**2


def edge_field_symbolic(edge):
    """
    Conservative nonlinear pairwise vector field for one edge.

    For edge (i,j):
        F_i = w_ij [phi(x_j) - phi(x_i)]
        F_j = -F_i
    """
    i, j = edge

    if edge not in edge_weight_exact:
        raise KeyError(f"Unknown microscopic edge: {edge}")

    w = edge_weight_exact[edge]

    field = sp.zeros(N, 1)

    interaction = w * (
        phi_symbolic(x_symbols[j - 1])
        -
        phi_symbolic(x_symbols[i - 1])
    )

    field[i - 1] += interaction
    field[j - 1] -= interaction

    return field


def edge_field_numeric(X, edge):
    """
    Vectorized numerical version.

    Parameters
    ----------
    X : ndarray, shape (..., N)
        One state or a batch of states.
    edge : tuple
        Frozen microscopic edge.

    Returns
    -------
    F : ndarray, same shape as X
    """
    i, j = edge

    if edge not in edge_weight:
        raise KeyError(f"Unknown microscopic edge: {edge}")

    w = edge_weight[edge]

    F = np.zeros_like(X, dtype=float)

    interaction = w * (
        phi_numeric(X[..., j - 1])
        -
        phi_numeric(X[..., i - 1])
    )

    F[..., i - 1] += interaction
    F[..., j - 1] -= interaction

    return F


# --------------------------------------------------------------
# Persistent native triadic interactions
#
# For h = {i,j,k},
#
# T_i =
#     g x_j x_k
#     - g/2 x_i x_j
#     - g/2 x_i x_k
#
# with cyclic permutations for j and k.
#
# This field is:
#   - quadratic
#   - symmetric under permutation of the three nodes
#   - conservative
#   - irreducibly triadic
# --------------------------------------------------------------

NATIVE_TRIADS = (
    (1, 2, 3),
    (2, 5, 8),
)

NATIVE_G = 0.02
NATIVE_G_EXACT = sp.Rational(1, 50)


def triad_field_symbolic(triad, g=NATIVE_G_EXACT):
    i, j, k = triad

    xi = x_symbols[i - 1]
    xj = x_symbols[j - 1]
    xk = x_symbols[k - 1]

    field = sp.zeros(N, 1)

    field[i - 1] += (
        g * xj * xk
        - g / 2 * xi * xj
        - g / 2 * xi * xk
    )

    field[j - 1] += (
        g * xi * xk
        - g / 2 * xj * xi
        - g / 2 * xj * xk
    )

    field[k - 1] += (
        g * xi * xj
        - g / 2 * xk * xi
        - g / 2 * xk * xj
    )

    return field


def triad_field_numeric(X, triad, g=NATIVE_G):
    """
    Vectorized native triadic field.
    """
    i, j, k = triad

    xi = X[..., i - 1]
    xj = X[..., j - 1]
    xk = X[..., k - 1]

    F = np.zeros_like(X, dtype=float)

    F[..., i - 1] += (
        g * xj * xk
        - 0.5 * g * xi * xj
        - 0.5 * g * xi * xk
    )

    F[..., j - 1] += (
        g * xi * xk
        - 0.5 * g * xj * xi
        - 0.5 * g * xj * xk
    )

    F[..., k - 1] += (
        g * xi * xj
        - 0.5 * g * xk * xi
        - 0.5 * g * xk * xj
    )

    return F


# --------------------------------------------------------------
# Precompute symbolic microscopic fields
# --------------------------------------------------------------

edge_fields_symbolic = {
    edge: edge_field_symbolic(edge)
    for edge in microscopic_edges
}

native_triad_fields_symbolic = {
    triad: triad_field_symbolic(triad)
    for triad in NATIVE_TRIADS
}


# --------------------------------------------------------------
# Basic physical consistency checks
# --------------------------------------------------------------

# Every microscopic field should conserve sum_i x_i.
for edge, field in edge_fields_symbolic.items():
    assert sp.simplify(sum(field)) == 0

for triad, field in native_triad_fields_symbolic.items():
    assert sp.simplify(sum(field)) == 0


# Numerical vectorization check
X_check = data_rng.uniform(
    -0.5,
    0.5,
    size=(16, N)
)

for edge in microscopic_edges:
    F_check = edge_field_numeric(X_check, edge)

    assert F_check.shape == X_check.shape

    assert np.max(
        np.abs(F_check.sum(axis=1))
    ) < 1e-14


for triad in NATIVE_TRIADS:
    F_check = triad_field_numeric(X_check, triad)

    assert F_check.shape == X_check.shape

    assert np.max(
        np.abs(F_check.sum(axis=1))
    ) < 1e-14


print("Microscopic vector fields initialized.")
print("  pairwise edge fields =", len(edge_fields_symbolic))
print("  persistent native triads =", NATIVE_TRIADS)
print("  native triad strength =", NATIVE_G)
print("  symbolic conservation: PASS")
print("  numerical conservation: PASS")

Microscopic vector fields initialized.
  pairwise edge fields = 12
  persistent native triads = ((1, 2, 3), (2, 5, 8))
  native triad strength = 0.02
  symbolic conservation: PASS
  numerical conservation: PASS


In [18]:
# ================================================================
# Cell 3 — Forward temporal generators
#
# Each snapshot contains:
#   4 temporally activated pairwise edge fields
#   + the same two persistent native triadic fields
#
# No reverse protocol is constructed anywhere in this notebook.
# ================================================================


# --------------------------------------------------------------
# Persistent native contribution
# --------------------------------------------------------------

native_total_symbolic = sum(
    native_triad_fields_symbolic.values(),
    sp.zeros(N, 1)
)


def native_total_numeric(X):
    F = np.zeros_like(X, dtype=float)

    for triad in NATIVE_TRIADS:
        F += triad_field_numeric(
            X,
            triad
        )

    return F


# --------------------------------------------------------------
# Symbolic snapshot generators
# --------------------------------------------------------------

snapshot_fields_symbolic = []


for snapshot in snapshots:

    pairwise_part = sum(
        (
            edge_fields_symbolic[edge]
            for edge in snapshot
        ),
        sp.zeros(N, 1)
    )

    full_field = pairwise_part + native_total_symbolic

    snapshot_fields_symbolic.append(
        sp.Matrix([
            sp.expand(component)
            for component in full_field
        ])
    )


# --------------------------------------------------------------
# Numerical/vectorized snapshot generator
# --------------------------------------------------------------

def snapshot_field_numeric(X, snapshot_index):
    """
    Evaluate one full snapshot generator on one state or a batch.

    Parameters
    ----------
    X : ndarray, shape (..., N)
    snapshot_index : int
        Python index 0,...,5.
    """

    snapshot = snapshots[snapshot_index]

    F = np.zeros_like(X, dtype=float)

    # Temporally activated pairwise sector
    for edge in snapshot:
        F += edge_field_numeric(
            X,
            edge
        )

    # Persistent native triads
    F += native_total_numeric(X)

    return F


# --------------------------------------------------------------
# Physical consistency audit
# --------------------------------------------------------------

assert len(snapshot_fields_symbolic) == len(snapshots)


# Symbolic conservation
for r, field in enumerate(
    snapshot_fields_symbolic,
    start=1
):
    assert sp.simplify(sum(field)) == 0


# Numerical conservation
X_check = data_rng.uniform(
    -0.5,
    0.5,
    size=(32, N)
)

max_conservation_error = 0.0

for r in range(len(snapshots)):

    F_check = snapshot_field_numeric(
        X_check,
        r
    )

    error = np.max(
        np.abs(
            F_check.sum(axis=1)
        )
    )

    max_conservation_error = max(
        max_conservation_error,
        error
    )


assert max_conservation_error < 1e-13


# --------------------------------------------------------------
# Persistent-exposure check
#
# Native HOIs are present in every snapshot, so their exposure
# over a full aggregation window is epsilon, not 6 epsilon.
# --------------------------------------------------------------

n_snapshots = len(snapshots)
snapshot_fraction = 1.0 / n_snapshots

assert n_snapshots == 6
assert np.isclose(
    n_snapshots * snapshot_fraction,
    1.0
)


print("Forward snapshot generators constructed.")
print("  number of snapshots =", n_snapshots)
print("  pairwise edges / snapshot =", 4)
print("  persistent native triads / snapshot =", len(NATIVE_TRIADS))
print("  snapshot duration = epsilon /", n_snapshots)
print(
    "  max numerical conservation error =",
    f"{max_conservation_error:.3e}"
)
print("  reverse protocol constructed = False")

Forward snapshot generators constructed.
  number of snapshots = 6
  pairwise edges / snapshot = 4
  persistent native triads / snapshot = 2
  snapshot duration = epsilon / 6
  max numerical conservation error = 3.331e-16
  reverse protocol constructed = False


In [19]:
# ================================================================
# Cell 4 — Forward-only endpoint data generation
#
# Observations:
#     X0
#     XF(epsilon)
#
# Same initial conditions are used for every epsilon.
# No reverse trajectories are generated.
# ================================================================


# --------------------------------------------------------------
# Dataset RNG
#
# Reinitialize explicitly so previous diagnostic random draws
# cannot change the formal dataset.
# --------------------------------------------------------------

DATA_SEED = SEED + 202
dataset_rng = np.random.default_rng(DATA_SEED)


# --------------------------------------------------------------
# Initial conditions
# --------------------------------------------------------------

X0 = dataset_rng.uniform(
    -0.5,
    0.5,
    size=(N_TOTAL, N)
)

X0_train = X0[:N_TRAIN]
X0_test = X0[N_TRAIN:]


# --------------------------------------------------------------
# Vectorized fixed-step RK4
# --------------------------------------------------------------

def rk4_integrate_snapshot(
    X,
    snapshot_index,
    duration,
    max_step=MAX_RK4_STEP,
):
    """
    Integrate one snapshot generator for a given duration.

    X can contain a batch of states with shape (n_samples, N).
    """

    n_steps = max(
        1,
        int(np.ceil(duration / max_step))
    )

    dt = duration / n_steps

    Y = np.array(
        X,
        dtype=float,
        copy=True
    )

    for _ in range(n_steps):

        k1 = snapshot_field_numeric(
            Y,
            snapshot_index
        )

        k2 = snapshot_field_numeric(
            Y + 0.5 * dt * k1,
            snapshot_index
        )

        k3 = snapshot_field_numeric(
            Y + 0.5 * dt * k2,
            snapshot_index
        )

        k4 = snapshot_field_numeric(
            Y + dt * k3,
            snapshot_index
        )

        Y += (
            dt / 6.0
        ) * (
            k1
            + 2.0 * k2
            + 2.0 * k3
            + k4
        )

    return Y


def forward_protocol(
    X_initial,
    epsilon,
):
    """
    Apply the six snapshots in chronological order.

        G1 -> G2 -> ... -> G6

    The full aggregation-window duration is epsilon.
    Each snapshot lasts epsilon / 6.
    """

    Y = np.array(
        X_initial,
        dtype=float,
        copy=True
    )

    snapshot_duration = (
        epsilon
        /
        len(snapshots)
    )

    for snapshot_index in range(
        len(snapshots)
    ):

        Y = rk4_integrate_snapshot(
            Y,
            snapshot_index,
            snapshot_duration,
        )

    return Y


# --------------------------------------------------------------
# Generate forward endpoint observations
# --------------------------------------------------------------

forward_data = {}

generation_rows = []


initial_mass = X0.sum(axis=1)


for epsilon in EPS_VALUES:

    XF = forward_protocol(
        X0,
        epsilon
    )

    forward_data[
        float(epsilon)
    ] = XF


    # ----------------------------------------------------------
    # Basic sanity diagnostics
    # ----------------------------------------------------------

    mass_error = np.max(
        np.abs(
            XF.sum(axis=1)
            -
            initial_mass
        )
    )

    displacement_rms = np.sqrt(
        np.mean(
            (XF - X0) ** 2
        )
    )

    generation_rows.append({
        "epsilon":
            float(epsilon),

        "displacement_rms":
            float(displacement_rms),

        "max_conservation_error":
            float(mass_error),

        "state_min":
            float(XF.min()),

        "state_max":
            float(XF.max()),
    })


generation_ledger = pd.DataFrame(
    generation_rows
)


# --------------------------------------------------------------
# Formal learner boundary
#
# Everything below the inference section should access the
# experiment only through learner_data.
# --------------------------------------------------------------

learner_data = {

    "epsilon_values":
        EPS_VALUES.copy(),

    "X0_train":
        X0_train.copy(),

    "X0_test":
        X0_test.copy(),

    "forward_train": {
        float(epsilon):
            forward_data[
                float(epsilon)
            ][:N_TRAIN].copy()

        for epsilon in EPS_VALUES
    },

    "forward_test": {
        float(epsilon):
            forward_data[
                float(epsilon)
            ][N_TRAIN:].copy()

        for epsilon in EPS_VALUES
    },
}


# --------------------------------------------------------------
# Final checks
# --------------------------------------------------------------

assert X0.shape == (
    N_TOTAL,
    N
)

assert X0_train.shape == (
    N_TRAIN,
    N
)

assert X0_test.shape == (
    N_TEST,
    N
)

assert len(
    forward_data
) == len(
    EPS_VALUES
)

assert generation_ledger[
    "max_conservation_error"
].max() < 1e-12


print("Forward-only dataset generated.")
print("  data seed =", DATA_SEED)
print("  total initial conditions =", N_TOTAL)
print("  train / test =", N_TRAIN, "/", N_TEST)
print("  resolutions =", len(EPS_VALUES))
print("  reverse trajectories generated = False")

print()
display(generation_ledger)

Forward-only dataset generated.
  data seed = 20261013
  total initial conditions = 4000
  train / test = 3000 / 1000
  resolutions = 11
  reverse trajectories generated = False



,epsilon,displacement_rms,max_conservation_error,state_min,state_max
0,0.0050,0.001756,4.440892e-16,-0.499023,0.498703
1,0.0075,0.002629,4.440892e-16,-0.498586,0.498155
2,0.0100,0.003498,8.881784e-16,-0.498439,0.497608
3,0.0150,0.005226,6.661338e-16,-0.498210,0.496933
4,0.0200,0.006940,6.661338e-16,-0.497980,0.496361
5,0.0300,0.010326,8.881784e-16,-0.497698,0.495214
6,0.0400,0.013658,8.881784e-16,-0.497481,0.494142
7,0.0600,0.020164,1.110223e-15,-0.497018,0.492671
8,0.0800,0.026465,1.554312e-15,-0.496518,0.491173
9,0.1000,0.032568,1.332268e-15,-0.495981,0.489650


Resolution preprocessing

In [6]:
# ================================================================
# Cell 5 — Resolution extrapolation from forward endpoint data
#
# For each initial condition:
#
#     Y(x, epsilon) = [x_F(epsilon) - x_0] / epsilon
#
# Fit locally in resolution:
#
#     Y = A + epsilon G1 + epsilon^2 G2 + epsilon^3 G3
#
# using the four finest aggregation windows.
#
# No dynamical dictionary is used in this cell.
# No microscopic information is used.
# ================================================================


# --------------------------------------------------------------
# Frozen extrapolation rule
# --------------------------------------------------------------

EPS_EXTRAP = np.array([
    0.005,
    0.0075,
    0.010,
    0.015,
], dtype=float)

EXTRAP_ORDER = 3

assert len(EPS_EXTRAP) == EXTRAP_ORDER + 1

for epsilon in EPS_EXTRAP:
    assert float(epsilon) in learner_data["forward_train"]


# --------------------------------------------------------------
# Build finite-window forward observable
#
# Y(epsilon) = [XF(epsilon) - X0] / epsilon
# --------------------------------------------------------------

def build_forward_observable(
    X0_local,
    forward_local,
    epsilon_values,
):
    """
    Returns
    -------
    Y : ndarray
        shape (n_epsilon, n_samples, N)
    """

    return np.stack([
        (
            forward_local[float(epsilon)]
            -
            X0_local
        )
        /
        float(epsilon)

        for epsilon in epsilon_values
    ], axis=0)


Y_train_eps = build_forward_observable(
    learner_data["X0_train"],
    learner_data["forward_train"],
    EPS_EXTRAP,
)

Y_test_eps = build_forward_observable(
    learner_data["X0_test"],
    learner_data["forward_test"],
    EPS_EXTRAP,
)


# --------------------------------------------------------------
# Numerically stable cubic fit in scaled resolution
#
# z = epsilon / h
#
# Y = c0 + c1 z + c2 z^2 + c3 z^3
#
# Therefore:
#
# A  = c0
# G1 = c1 / h
# --------------------------------------------------------------

EPS_SCALE = float(
    EPS_EXTRAP.max()
)

z_values = (
    EPS_EXTRAP
    /
    EPS_SCALE
)


resolution_design = np.column_stack([
    z_values ** power
    for power in range(
        EXTRAP_ORDER + 1
    )
])


resolution_design_inv = np.linalg.inv(
    resolution_design
)


print(
    "Resolution design condition number:",
    np.linalg.cond(
        resolution_design
    )
)


def extrapolate_resolution_targets(
    Y_eps
):
    """
    Parameters
    ----------
    Y_eps : ndarray
        shape (n_epsilon, n_samples, N)

    Returns
    -------
    A_data : ndarray
        Estimated epsilon -> 0 intercept.

    G1_data : ndarray
        Estimated physical derivative dY/depsilon at epsilon = 0.

    higher_coefficients : ndarray
        Coefficients in the scaled z basis.
    """

    # shape:
    # (4,4) @ (4,n_samples,N)
    # -> (4,n_samples,N)

    coeff_scaled = np.einsum(
        "ab,bni->ani",
        resolution_design_inv,
        Y_eps,
    )

    A_data = (
        coeff_scaled[0]
    )

    G1_data = (
        coeff_scaled[1]
        /
        EPS_SCALE
    )

    return (
        A_data,
        G1_data,
        coeff_scaled,
    )


(
    A_train_data,
    G1_train_data,
    resolution_coeff_train,
) = extrapolate_resolution_targets(
    Y_train_eps
)


(
    A_test_data,
    G1_test_data,
    resolution_coeff_test,
) = extrapolate_resolution_targets(
    Y_test_eps
)


# --------------------------------------------------------------
# Data-only sanity checks
# --------------------------------------------------------------

def rms(X):
    return float(
        np.sqrt(
            np.mean(
                X ** 2
            )
        )
    )


# Conservation implies sum_i F_i = 0.
# Since extrapolation is linear, the inferred targets should
# inherit this property numerically.

A_train_conservation = np.max(
    np.abs(
        A_train_data.sum(axis=1)
    )
)

G1_train_conservation = np.max(
    np.abs(
        G1_train_data.sum(axis=1)
    )
)


# Reconstruct the four input resolutions.
# Since this is cubic interpolation through four points,
# this should be at floating-point precision.

Y_train_reconstructed = np.einsum(
    "ea,ani->eni",
    resolution_design,
    resolution_coeff_train,
)

interpolation_rms = rms(
    Y_train_reconstructed
    -
    Y_train_eps
)


print()
print("Resolution extrapolation complete.")
print("  epsilon values =", EPS_EXTRAP)
print("  epsilon scale =", EPS_SCALE)
print("  train target shape =", A_train_data.shape)
print("  test target shape =", A_test_data.shape)

print()
print("Target magnitudes:")
print(
    "  RMS(A_data)  =",
    f"{rms(A_train_data):.8f}"
)
print(
    "  RMS(G1_data) =",
    f"{rms(G1_train_data):.8f}"
)

print()
print("Conservation:")
print(
    "  max |sum_i A_i|  =",
    f"{A_train_conservation:.3e}"
)
print(
    "  max |sum_i G1_i| =",
    f"{G1_train_conservation:.3e}"
)

print()
print(
    "Cubic interpolation RMS =",
    f"{interpolation_rms:.3e}"
)

Resolution design condition number: 642.8653929148553

Resolution extrapolation complete.
  epsilon values = [0.005  0.0075 0.01   0.015 ]
  epsilon scale = 0.015
  train target shape = (3000, 8)
  test target shape = (1000, 8)

Target magnitudes:
  RMS(A_data)  = 0.35231606
  RMS(G1_data) = 0.29990084

Conservation:
  max |sum_i A_i|  = 1.245e-12
  max |sum_i G1_i| = 4.338e-10

Cubic interpolation RMS = 3.459e-15


Adaptive Group LASSO

In [7]:
# ================================================================
# Cell 6 — Generic polynomial library and structural groups
#
# Hypothesis class:
#     polynomial degree <= 3
#
# Structural group of coefficient (feature alpha, output i):
#
#     sigma(i, alpha)
#         = {output node i}
#           union
#           {nodes appearing in monomial alpha}
#
# All coefficients with the same sigma are assigned to one
# global structural group.
#
# No microscopic graph information is used.
# No oracle support information is used.
# ================================================================


# --------------------------------------------------------------
# 1. Generic monomial library
#
# All nonconstant monomials in N variables with total degree 1..3.
# For N=8:
#
#     degree 1:   8
#     degree 2:  36
#     degree 3: 120
#
#     total = 164 features per output equation.
# --------------------------------------------------------------

def generate_monomial_exponents(
    n_variables,
    max_degree,
):
    """
    Return exponent tuples for every nonconstant monomial
    of total degree 1,...,max_degree.

    Example:
        x1*x3^2  ->  (1,0,2,0,...)
    """

    exponent_list = []

    for degree in range(
        1,
        max_degree + 1
    ):

        for variable_tuple in combinations_with_replacement(
            range(n_variables),
            degree
        ):

            exponent = [0] * n_variables

            for variable_index in variable_tuple:
                exponent[variable_index] += 1

            exponent_list.append(
                tuple(exponent)
            )

    return tuple(
        exponent_list
    )


monomial_exponents = generate_monomial_exponents(
    N,
    MAX_POLY_DEGREE,
)

N_FEATURES = len(
    monomial_exponents
)


assert N_FEATURES == 164


# --------------------------------------------------------------
# Human-readable monomial labels
# --------------------------------------------------------------

def monomial_label(exponent):
    pieces = []

    for j, power in enumerate(
        exponent,
        start=1
    ):

        if power == 0:
            continue

        if power == 1:
            pieces.append(
                f"x{j}"
            )
        else:
            pieces.append(
                f"x{j}^{power}"
            )

    return "*".join(
        pieces
    )


monomial_labels = tuple(
    monomial_label(exp)
    for exp in monomial_exponents
)


# --------------------------------------------------------------
# 2. Evaluate polynomial design matrix
# --------------------------------------------------------------

def evaluate_library(
    X,
    exponent_list=monomial_exponents,
):
    """
    Parameters
    ----------
    X : ndarray, shape (n_samples, N)

    Returns
    -------
    Theta : ndarray, shape (n_samples, N_FEATURES)
    """

    n_samples = X.shape[0]

    Theta = np.ones(
        (
            n_samples,
            len(exponent_list)
        ),
        dtype=float
    )

    for feature_index, exponent in enumerate(
        exponent_list
    ):

        column = np.ones(
            n_samples,
            dtype=float
        )

        for variable_index, power in enumerate(
            exponent
        ):

            if power:
                column *= (
                    X[:, variable_index]
                    ** power
                )

        Theta[:, feature_index] = column

    return Theta


Theta_train_raw = evaluate_library(
    learner_data["X0_train"]
)

Theta_test_raw = evaluate_library(
    learner_data["X0_test"]
)


assert Theta_train_raw.shape == (
    N_TRAIN,
    N_FEATURES
)

assert Theta_test_raw.shape == (
    N_TEST,
    N_FEATURES
)


# --------------------------------------------------------------
# 3. Column scaling
#
# Do NOT center columns:
# centering would destroy the direct polynomial/support meaning.
#
# Scale each feature to unit RMS on the training set.
# --------------------------------------------------------------

feature_scale = np.sqrt(
    np.mean(
        Theta_train_raw ** 2,
        axis=0
    )
)

assert np.all(
    feature_scale > 0
)


Theta_train = (
    Theta_train_raw
    /
    feature_scale
)

Theta_test = (
    Theta_test_raw
    /
    feature_scale
)


# --------------------------------------------------------------
# 4. Structural support of each coefficient direction
#
# Coefficient matrix convention:
#
#     B[feature_index, output_index]
#
# Flattening convention:
#
#     flat_index
#         = feature_index * N + output_index
# --------------------------------------------------------------

def variable_support_from_exponent(
    exponent
):
    """
    1-indexed node support appearing in the monomial.
    """

    return tuple(
        j + 1

        for j, power in enumerate(
            exponent
        )

        if power > 0
    )


def coefficient_flat_index(
    feature_index,
    output_index
):
    return (
        feature_index * N
        +
        output_index
    )


coefficient_rows = []

structural_groups = defaultdict(
    list
)


for feature_index, exponent in enumerate(
    monomial_exponents
):

    variable_support = set(
        variable_support_from_exponent(
            exponent
        )
    )

    degree = sum(
        exponent
    )

    for output_index in range(N):

        output_node = (
            output_index + 1
        )

        structural_support = tuple(
            sorted(
                variable_support
                |
                {output_node}
            )
        )

        support_size = len(
            structural_support
        )

        # degree <= 3 implies support <= 4
        assert (
            support_size
            <= MAX_SUPPORT_SIZE
        )

        flat_index = coefficient_flat_index(
            feature_index,
            output_index
        )

        structural_groups[
            structural_support
        ].append(
            flat_index
        )

        coefficient_rows.append({
            "flat_index":
                flat_index,

            "feature_index":
                feature_index,

            "output_index":
                output_index,

            "output_node":
                output_node,

            "monomial":
                monomial_labels[
                    feature_index
                ],

            "degree":
                degree,

            "variable_support":
                tuple(
                    sorted(
                        variable_support
                    )
                ),

            "structural_support":
                structural_support,

            "support_size":
                support_size,
        })


coefficient_ledger = pd.DataFrame(
    coefficient_rows
)


# Convert lists to immutable arrays
structural_groups = {
    support:
        np.array(
            indices,
            dtype=int
        )

    for support, indices
    in structural_groups.items()
}


N_COEFFICIENTS = (
    N_FEATURES * N
)

N_GROUPS = len(
    structural_groups
)


assert len(
    coefficient_ledger
) == N_COEFFICIENTS

assert N_COEFFICIENTS == 1312

assert N_GROUPS == 162


# --------------------------------------------------------------
# 5. Verify that groups are disjoint and exhaustive
# --------------------------------------------------------------

all_group_indices = np.concatenate(
    list(
        structural_groups.values()
    )
)

assert len(
    all_group_indices
) == N_COEFFICIENTS

assert len(
    np.unique(
        all_group_indices
    )
) == N_COEFFICIENTS

assert set(
    all_group_indices.tolist()
) == set(
    range(
        N_COEFFICIENTS
    )
)


# --------------------------------------------------------------
# 6. Structural summary
# --------------------------------------------------------------

group_rows = []

for support, indices in sorted(
    structural_groups.items(),
    key=lambda item: (
        len(item[0]),
        item[0]
    )
):

    group_rows.append({
        "support":
            support,

        "support_size":
            len(support),

        "group_size":
            len(indices),
    })


group_ledger = pd.DataFrame(
    group_rows
)


coefficient_count_by_support_size = (
    coefficient_ledger
    .groupby("support_size")
    .size()
    .to_dict()
)

group_count_by_support_size = (
    group_ledger
    .groupby("support_size")
    .size()
    .to_dict()
)


# Frozen combinatorial checks
assert coefficient_count_by_support_size == {
    1: 24,
    2: 336,
    3: 672,
    4: 280,
}

assert group_count_by_support_size == {
    1: 8,
    2: 28,
    3: 56,
    4: 70,
}


# --------------------------------------------------------------
# 7. Design identifiability audit
#
# Same scalar polynomial library is shared by all output equations.
# --------------------------------------------------------------

library_rank = np.linalg.matrix_rank(
    Theta_train
)

library_condition = np.linalg.cond(
    Theta_train
)


assert library_rank == N_FEATURES


# --------------------------------------------------------------
# 8. Summary
# --------------------------------------------------------------

print("Generic structural library constructed.")

print()
print("Polynomial features:")
print(
    "  features / output =",
    N_FEATURES
)
print(
    "  total coefficient directions =",
    N_COEFFICIENTS
)

print()
print("Structural groups:")
print(
    "  total groups =",
    N_GROUPS
)

print(
    "  group counts by support size =",
    group_count_by_support_size
)

print(
    "  coefficient counts by support size =",
    coefficient_count_by_support_size
)

print()
print("Training design:")
print(
    "  shape =",
    Theta_train.shape
)
print(
    "  rank =",
    library_rank
)
print(
    "  normalized condition number =",
    f"{library_condition:.6f}"
)

print()
print("Sample structural groups:")
display(
    group_ledger.head(15)
)

Generic structural library constructed.

Polynomial features:
  features / output = 164
  total coefficient directions = 1312

Structural groups:
  total groups = 162
  group counts by support size = {1: 8, 2: 28, 3: 56, 4: 70}
  coefficient counts by support size = {1: 24, 2: 336, 3: 672, 4: 280}

Training design:
  shape = (3000, 164)
  rank = 164
  normalized condition number = 12.395993

Sample structural groups:


,support,support_size,group_size
0,"(1,)",1,3
1,"(2,)",1,3
2,"(3,)",1,3
3,"(4,)",1,3
4,"(5,)",1,3
5,"(6,)",1,3
6,"(7,)",1,3
7,"(8,)",1,3
8,"(1, 2)",2,12
9,"(1, 3)",2,12


In [8]:
# ================================================================
# Cell 7 — Adaptive Group-LASSO initialization
#
# Target:
#     A_data(x)  ~  F^(0)(x)
#
# Steps:
#   1. OLS initial estimator
#   2. group norms
#   3. adaptive weights
#   4. weighted lambda_max
#   5. frozen lambda path
#
# No oracle information is used.
# ================================================================


# --------------------------------------------------------------
# 1. Initial estimator
#
# Because N_TRAIN >> N_FEATURES here, ordinary least squares
# gives a consistent initial estimate for Adaptive Group LASSO.
#
# B convention:
#     shape = (N_FEATURES, N)
#
# Theta_train @ B  ->  vector field, shape (n_samples, N)
# --------------------------------------------------------------

B0_init_scaled, *_ = np.linalg.lstsq(
    Theta_train,
    A_train_data,
    rcond=None,
)


assert B0_init_scaled.shape == (
    N_FEATURES,
    N,
)


A_train_init = (
    Theta_train
    @ B0_init_scaled
)

A_test_init = (
    Theta_test
    @ B0_init_scaled
)


def relative_field_error(
    prediction,
    target,
):
    return float(
        np.linalg.norm(
            prediction - target
        )
        /
        np.linalg.norm(
            target
        )
    )


init_train_relerr = relative_field_error(
    A_train_init,
    A_train_data,
)

init_test_relerr = relative_field_error(
    A_test_init,
    A_test_data,
)


# --------------------------------------------------------------
# 2. Initial structural group norms
#
# Flattening is C-order:
#
#     flat_index = feature_index * N + output_index
#
# exactly matching Cell 6.
# --------------------------------------------------------------

beta0_init_flat = (
    B0_init_scaled
    .ravel(order="C")
)


initial_group_norms = {}

for support, indices in structural_groups.items():

    initial_group_norms[support] = float(
        np.linalg.norm(
            beta0_init_flat[
                indices
            ]
        )
    )


group_norm_values = np.array(
    list(
        initial_group_norms.values()
    )
)


# --------------------------------------------------------------
# 3. Adaptive weights
#
# Standard adaptive Group-LASSO form:
#
#     w_S =
#       sqrt(|g_S|)
#       /
#       ( ||beta_init,S||_2 + delta )^gamma
#
# gamma = 1.
#
# delta is only a numerical floor, fixed relative to the
# largest initial group norm.
# --------------------------------------------------------------

ADAPTIVE_GAMMA = 1.0

ADAPTIVE_DELTA = max(
    1e-12,
    1e-8 * group_norm_values.max(),
)


adaptive_group_weights = {}

for support, indices in structural_groups.items():

    norm_init = initial_group_norms[
        support
    ]

    adaptive_group_weights[
        support
    ] = (
        np.sqrt(
            len(indices)
        )
        /
        (
            norm_init
            +
            ADAPTIVE_DELTA
        )
        ** ADAPTIVE_GAMMA
    )


weight_values = np.array(
    list(
        adaptive_group_weights.values()
    )
)


# --------------------------------------------------------------
# 4. Lambda_max
#
# Objective convention used from now on:
#
#   min_B
#
#       1/(2 n) ||A - Theta B||_F^2
#
#       + lambda *
#         sum_S w_S ||B_S||_2
#
#
# At B = 0:
#
#   grad = -(Theta^T A) / n
#
# lambda_max is the smallest lambda for which B=0 satisfies
# every weighted group KKT condition.
# --------------------------------------------------------------

gradient_at_zero = (
    -Theta_train.T
    @ A_train_data
    /
    N_TRAIN
)


gradient_zero_flat = (
    gradient_at_zero
    .ravel(order="C")
)


lambda_candidates = []

for support, indices in structural_groups.items():

    grad_group_norm = np.linalg.norm(
        gradient_zero_flat[
            indices
        ]
    )

    weight = adaptive_group_weights[
        support
    ]

    lambda_candidates.append(
        grad_group_norm
        /
        weight
    )


LAMBDA_MAX_F0 = float(
    np.max(
        lambda_candidates
    )
)


# --------------------------------------------------------------
# 5. Frozen lambda path
#
# Descending path:
#
#     lambda_max
#       -> 1e-6 * lambda_max
#
# 80 logarithmically spaced points.
#
# The model will NOT be selected by an oracle support count.
# Cell 8 will use refitted BIC along this path.
# --------------------------------------------------------------

N_LAMBDAS_F0 = 80
LAMBDA_MIN_RATIO_F0 = 1e-6


lambda_path_F0 = (
    LAMBDA_MAX_F0
    *
    np.geomspace(
        1.0,
        LAMBDA_MIN_RATIO_F0,
        N_LAMBDAS_F0,
    )
)


assert np.all(
    np.diff(
        lambda_path_F0
    ) < 0
)


# --------------------------------------------------------------
# 6. Diagnostic ledger
#
# Still entirely non-oracle.
# --------------------------------------------------------------

adaptive_rows = []

for support in sorted(
    structural_groups,
    key=lambda s: (
        len(s),
        s
    )
):

    adaptive_rows.append({
        "support":
            support,

        "support_size":
            len(support),

        "group_size":
            len(
                structural_groups[
                    support
                ]
            ),

        "initial_norm":
            initial_group_norms[
                support
            ],

        "adaptive_weight":
            adaptive_group_weights[
                support
            ],
    })


adaptive_weight_ledger = pd.DataFrame(
    adaptive_rows
)


print("Adaptive Group-LASSO initialization complete.")

print()
print("Initial OLS fit:")
print(
    "  train relative field error =",
    f"{init_train_relerr:.3e}"
)
print(
    "  test relative field error  =",
    f"{init_test_relerr:.3e}"
)

print()
print("Initial group norms:")
print(
    "  min =",
    f"{group_norm_values.min():.3e}"
)
print(
    "  median =",
    f"{np.median(group_norm_values):.3e}"
)
print(
    "  max =",
    f"{group_norm_values.max():.3e}"
)

print()
print("Adaptive weights:")
print(
    "  gamma =",
    ADAPTIVE_GAMMA
)
print(
    "  delta =",
    f"{ADAPTIVE_DELTA:.3e}"
)
print(
    "  min =",
    f"{weight_values.min():.3e}"
)
print(
    "  median =",
    f"{np.median(weight_values):.3e}"
)
print(
    "  max =",
    f"{weight_values.max():.3e}"
)

print()
print("Regularization path:")
print(
    "  lambda_max =",
    f"{LAMBDA_MAX_F0:.6e}"
)
print(
    "  lambda_min =",
    f"{lambda_path_F0[-1]:.6e}"
)
print(
    "  number of lambdas =",
    N_LAMBDAS_F0
)

print()
print("Largest initial structural groups:")
display(
    adaptive_weight_ledger
    .sort_values(
        "initial_norm",
        ascending=False
    )
    .head(20)
)

Adaptive Group-LASSO initialization complete.

Initial OLS fit:
  train relative field error = 1.430e-10
  test relative field error  = 1.553e-10

Initial group norms:
  min = 4.579e-13
  median = 1.341e-11
  max = 4.120e-01

Adaptive weights:
  gamma = 1.0
  delta = 4.120e-09
  min = 4.204e+00
  median = 4.852e+08
  max = 8.402e+08

Regularization path:
  lambda_max = 1.315276e-01
  lambda_min = 1.315276e-07
  number of lambdas = 80

Largest initial structural groups:


,support,support_size,group_size,initial_norm,adaptive_weight
4,"(5,)",1,3,0.411975,4.204261
2,"(3,)",1,3,0.406085,4.265240
6,"(7,)",1,3,0.299638,5.780471
3,"(4,)",1,3,0.298433,5.803826
0,"(1,)",1,3,0.293608,5.899188
1,"(2,)",1,3,0.284921,6.079058
5,"(6,)",1,3,0.225209,7.690860
7,"(8,)",1,3,0.183168,9.456099
30,"(5, 6)",2,12,0.163427,21.196639
33,"(6, 7)",2,12,0.154916,22.361200


In [9]:
# # ================================================================
# # Cell 8 — Adaptive Group LASSO path for F^(0)
# #
# # Solve:
# #
# #   min_B  1/(2n) ||A_data - Theta B||_F^2
# #
# #          + lambda * sum_S w_S ||B_S||_2
# #
# # using proximal gradient with warm starts.
# #
# # For every distinct active support model:
# #   - unpenalized OLS refit
# #   - compute training RSS
# #   - compute BIC
# #
# # BIC selects the final model.
# #
# # No oracle information is used.
# # ================================================================


# # --------------------------------------------------------------
# # 1. Smooth-loss Lipschitz constant
# #
# # Gradient:
# #
# #   grad_B =
# #       Theta^T (Theta B - Y) / n
# #
# # Lipschitz constant:
# #
# #   L = ||Theta||_2^2 / n
# # --------------------------------------------------------------

# spectral_norm_theta = np.linalg.norm(
#     Theta_train,
#     ord=2
# )

# L_F0 = (
#     spectral_norm_theta ** 2
#     /
#     N_TRAIN
# )

# STEP_F0 = 1.0 / L_F0


# print(
#     "Gradient Lipschitz constant =",
#     f"{L_F0:.6f}"
# )

# print(
#     "Proximal step size =",
#     f"{STEP_F0:.6f}"
# )


# # --------------------------------------------------------------
# # 2. Weighted group proximal operator
# # --------------------------------------------------------------

# def adaptive_group_prox(
#     B,
#     threshold_lambda,
#     group_weights,
# ):
#     """
#     Proximal operator for

#         threshold_lambda *
#         sum_S w_S ||B_S||_2

#     Groups are disjoint and exhaustive.
#     """

#     beta_flat = B.ravel(
#         order="C"
#     )

#     output_flat = beta_flat.copy()

#     for support, indices in structural_groups.items():

#         group_vector = beta_flat[
#             indices
#         ]

#         group_norm = np.linalg.norm(
#             group_vector
#         )

#         threshold = (
#             threshold_lambda
#             *
#             group_weights[
#                 support
#             ]
#         )

#         if group_norm <= threshold:

#             output_flat[
#                 indices
#             ] = 0.0

#         else:

#             shrinkage = (
#                 1.0
#                 -
#                 threshold
#                 /
#                 group_norm
#             )

#             output_flat[
#                 indices
#             ] *= shrinkage

#     return output_flat.reshape(
#         B.shape,
#         order="C"
#     )


# # --------------------------------------------------------------
# # 3. Proximal-gradient solver
# #
# # Warm starts are used along descending lambda path.
# # --------------------------------------------------------------

# def fit_adaptive_group_lasso(
#     Theta,
#     Y,
#     lambda_value,
#     group_weights,
#     B_start=None,
#     step_size=STEP_F0,
#     max_iter=20000,
#     tol=1e-10,
# ):
#     """
#     Basic proximal-gradient solver.

#     Returns
#     -------
#     B : ndarray
#     info : dict
#     """

#     n_samples, n_features = Theta.shape
#     n_outputs = Y.shape[1]

#     if B_start is None:

#         B = np.zeros(
#             (
#                 n_features,
#                 n_outputs
#             ),
#             dtype=float
#         )

#     else:

#         B = np.array(
#             B_start,
#             dtype=float,
#             copy=True
#         )


#     for iteration in range(
#         1,
#         max_iter + 1
#     ):

#         residual = (
#             Theta @ B
#             -
#             Y
#         )

#         gradient = (
#             Theta.T
#             @ residual
#             /
#             n_samples
#         )

#         B_new = adaptive_group_prox(
#             B - step_size * gradient,
#             step_size * lambda_value,
#             group_weights,
#         )


#         change = np.linalg.norm(
#             B_new - B
#         )

#         reference = max(
#             1.0,
#             np.linalg.norm(B)
#         )

#         relative_change = (
#             change
#             /
#             reference
#         )

#         B = B_new


#         if relative_change < tol:
#             break


#     return B, {
#         "iterations":
#             iteration,

#         "relative_change":
#             relative_change,

#         "converged":
#             relative_change < tol,
#     }


# # --------------------------------------------------------------
# # 4. Active structural support readout
# # --------------------------------------------------------------

# ACTIVE_GROUP_TOL = 1e-10


# def active_supports_from_B(
#     B,
#     tol=ACTIVE_GROUP_TOL,
# ):
#     beta_flat = B.ravel(
#         order="C"
#     )

#     active = []

#     for support, indices in structural_groups.items():

#         group_norm = np.linalg.norm(
#             beta_flat[
#                 indices
#             ]
#         )

#         if group_norm > tol:
#             active.append(
#                 support
#             )

#     return tuple(
#         sorted(
#             active,
#             key=lambda s: (
#                 len(s),
#                 s
#             )
#         )
#     )


# # --------------------------------------------------------------
# # 5. Unpenalized OLS refit on selected support model
# #
# # Because the same scalar dictionary is used for each output,
# # refitting can be done output-by-output without constructing
# # a huge block-diagonal matrix.
# # --------------------------------------------------------------

# def refit_active_model(
#     Theta,
#     Y,
#     active_supports,
# ):
#     """
#     OLS debiasing on the coefficient directions belonging to
#     the selected structural groups.
#     """

#     B_refit = np.zeros(
#         (
#             Theta.shape[1],
#             Y.shape[1]
#         ),
#         dtype=float
#     )

#     active_set = set(
#         active_supports
#     )


#     # Which features are active for each output equation?
#     for output_index in range(
#         Y.shape[1]
#     ):

#         active_features = []

#         for feature_index in range(
#             Theta.shape[1]
#         ):

#             flat_index = (
#                 feature_index * N
#                 +
#                 output_index
#             )

#             support = coefficient_ledger.loc[
#                 flat_index,
#                 "structural_support"
#             ]

#             if support in active_set:
#                 active_features.append(
#                     feature_index
#                 )


#         if len(active_features) == 0:
#             continue


#         active_features = np.array(
#             active_features,
#             dtype=int
#         )


#         coefficients, *_ = np.linalg.lstsq(
#             Theta[
#                 :,
#                 active_features
#             ],
#             Y[
#                 :,
#                 output_index
#             ],
#             rcond=None,
#         )


#         B_refit[
#             active_features,
#             output_index
#         ] = coefficients


#     return B_refit


# # --------------------------------------------------------------
# # 6. Model diagnostics and BIC
# #
# # Treat all node-component observations as scalar observations:
# #
# #     n_obs = N_TRAIN * N
# #
# # Number of free parameters is the number of coefficient
# # directions included by the selected groups.
# # --------------------------------------------------------------

# N_OBS_F0 = (
#     N_TRAIN * N
# )


# def model_statistics(
#     Theta,
#     Y,
#     B_refit,
#     active_supports,
# ):

#     prediction = (
#         Theta
#         @ B_refit
#     )

#     residual = (
#         Y
#         -
#         prediction
#     )

#     rss = float(
#         np.sum(
#             residual ** 2
#         )
#     )


#     n_parameters = int(
#         sum(
#             len(
#                 structural_groups[
#                     support
#                 ]
#             )

#             for support in active_supports
#         )
#     )


#     # Numerical safety only.
#     rss_safe = max(
#         rss,
#         np.finfo(float).tiny
#     )


#     bic = (
#         N_OBS_F0
#         *
#         np.log(
#             rss_safe
#             /
#             N_OBS_F0
#         )
#         +
#         n_parameters
#         *
#         np.log(
#             N_OBS_F0
#         )
#     )


#     relative_error = (
#         np.linalg.norm(
#             residual
#         )
#         /
#         np.linalg.norm(
#             Y
#         )
#     )


#     return {
#         "rss":
#             rss,

#         "bic":
#             float(bic),

#         "n_parameters":
#             n_parameters,

#         "relative_error":
#             float(relative_error),
#     }


# # --------------------------------------------------------------
# # 7. Run descending lambda path
# # --------------------------------------------------------------

# path_rows_F0 = []

# path_models_F0 = {}

# B_warm = np.zeros(
#     (
#         N_FEATURES,
#         N
#     ),
#     dtype=float
# )


# previous_model = None


# for path_index, lambda_value in enumerate(
#     lambda_path_F0
# ):

#     B_penalized, solver_info = (
#         fit_adaptive_group_lasso(
#             Theta_train,
#             A_train_data,
#             lambda_value,
#             adaptive_group_weights,
#             B_start=B_warm,
#         )
#     )

#     B_warm = B_penalized


#     active_supports = (
#         active_supports_from_B(
#             B_penalized
#         )
#     )


#     model_key = active_supports


#     # Only refit a structural model the first time it appears.
#     if model_key not in path_models_F0:

#         B_refit = refit_active_model(
#             Theta_train,
#             A_train_data,
#             active_supports,
#         )

#         stats = model_statistics(
#             Theta_train,
#             A_train_data,
#             B_refit,
#             active_supports,
#         )


#         path_models_F0[
#             model_key
#         ] = {
#             "B_refit":
#                 B_refit,

#             "stats":
#                 stats,
#         }

#     else:

#         B_refit = path_models_F0[
#             model_key
#         ][
#             "B_refit"
#         ]

#         stats = path_models_F0[
#             model_key
#         ][
#             "stats"
#         ]


#     path_rows_F0.append({
#         "path_index":
#             path_index,

#         "lambda":
#             float(
#                 lambda_value
#             ),

#         "n_groups":
#             len(
#                 active_supports
#             ),

#         "n_parameters":
#             stats[
#                 "n_parameters"
#             ],

#         "train_relative_error":
#             stats[
#                 "relative_error"
#             ],

#         "rss":
#             stats[
#                 "rss"
#             ],

#         "bic":
#             stats[
#                 "bic"
#             ],

#         "solver_iterations":
#             solver_info[
#                 "iterations"
#             ],

#         "solver_converged":
#             solver_info[
#                 "converged"
#             ],

#         "solver_relative_change":
#             solver_info[
#                 "relative_change"
#             ],

#         "active_supports":
#             active_supports,
#     })


# path_ledger_F0 = pd.DataFrame(
#     path_rows_F0
# )


# # --------------------------------------------------------------
# # 8. Select model by minimum BIC
# # --------------------------------------------------------------

# best_row_index_F0 = (
#     path_ledger_F0[
#         "bic"
#     ].idxmin()
# )


# best_row_F0 = (
#     path_ledger_F0.loc[
#         best_row_index_F0
#     ]
# )


# selected_supports_F0 = (
#     best_row_F0[
#         "active_supports"
#     ]
# )


# B0_hat_scaled = (
#     path_models_F0[
#         selected_supports_F0
#     ][
#         "B_refit"
#     ]
# )


# # --------------------------------------------------------------
# # 9. Held-out evaluation
# # --------------------------------------------------------------

# A_train_hat = (
#     Theta_train
#     @ B0_hat_scaled
# )

# A_test_hat = (
#     Theta_test
#     @ B0_hat_scaled
# )


# F0_train_relerr = relative_field_error(
#     A_train_hat,
#     A_train_data,
# )

# F0_test_relerr = relative_field_error(
#     A_test_hat,
#     A_test_data,
# )


# # --------------------------------------------------------------
# # 10. Structural summary
# # --------------------------------------------------------------

# selected_count_by_size_F0 = Counter(
#     len(support)
#     for support in selected_supports_F0
# )


# print()
# print("Adaptive Group LASSO F0 selection complete.")

# print()
# print("Path:")
# print(
#     "  lambda points =",
#     len(
#         lambda_path_F0
#     )
# )
# print(
#     "  distinct structural models =",
#     len(
#         path_models_F0
#     )
# )
# print(
#     "  all solver runs converged =",
#     bool(
#         path_ledger_F0[
#             "solver_converged"
#         ].all()
#     )
# )

# print()
# print("BIC-selected model:")
# print(
#     "  lambda =",
#     f"{best_row_F0['lambda']:.6e}"
# )
# print(
#     "  active groups =",
#     len(
#         selected_supports_F0
#     )
# )
# print(
#     "  active coefficients =",
#     int(
#         best_row_F0[
#             "n_parameters"
#         ]
#     )
# )
# print(
#     "  groups by support size =",
#     dict(
#         sorted(
#             selected_count_by_size_F0.items()
#         )
#     )
# )

# print()
# print("Refitted field error:")
# print(
#     "  train =",
#     f"{F0_train_relerr:.3e}"
# )
# print(
#     "  test  =",
#     f"{F0_test_relerr:.3e}"
# )

# print()
# print("Selected supports:")
# for support in selected_supports_F0:
#     print(
#         " ",
#         support
#     )

# print()
# print("Best BIC neighborhood:")
# display(
#     path_ledger_F0[
#         [
#             "path_index",
#             "lambda",
#             "n_groups",
#             "n_parameters",
#             "train_relative_error",
#             "bic",
#             "solver_iterations",
#             "solver_converged",
#         ]
#     ]
#     .sort_values(
#         "bic"
#     )
#     .head(12)
# )

Gradient Lipschitz constant = 6.827258
Proximal step size = 0.146472

Adaptive Group LASSO F0 selection complete.

Path:
  lambda points = 80
  distinct structural models = 12
  all solver runs converged = True

BIC-selected model:
  lambda = 3.755891e-07
  active groups = 25
  active coefficients = 228
  groups by support size = {1: 8, 2: 15, 3: 2}

Refitted field error:
  train = 7.493e-10
  test  = 7.791e-10

Selected supports:
  (1,)
  (2,)
  (3,)
  (4,)
  (5,)
  (6,)
  (7,)
  (8,)
  (1, 2)
  (1, 3)
  (1, 8)
  (2, 3)
  (2, 4)
  (2, 5)
  (2, 8)
  (3, 4)
  (3, 5)
  (4, 5)
  (5, 6)
  (5, 7)
  (5, 8)
  (6, 7)
  (7, 8)
  (1, 2, 3)
  (2, 5, 8)

Best BIC neighborhood:


,path_index,lambda,n_groups,n_parameters,train_relative_error,bic,solver_iterations,solver_converged
78,78,1.566629e-07,25,228,7.492688e-10,-1.056348e+06,787,True
73,73,3.755891e-07,25,228,7.492688e-10,-1.056348e+06,812,True
75,75,2.647368e-07,25,228,7.492688e-10,-1.056348e+06,826,True
74,74,3.153288e-07,25,228,7.492688e-10,-1.056348e+06,839,True
79,79,1.315276e-07,25,228,7.492688e-10,-1.056348e+06,774,True
77,77,1.866017e-07,25,228,7.492688e-10,-1.056348e+06,800,True
76,76,2.222618e-07,25,228,7.492688e-10,-1.056348e+06,813,True
63,63,2.158772e-06,22,192,2.046906e-03,-3.453268e+05,959,True
64,64,1.812414e-06,22,192,2.046906e-03,-3.453268e+05,940,True
65,65,1.521626e-06,22,192,2.046906e-03,-3.453268e+05,924,True


In [10]:
# ================================================================
# Cell 8 — Adaptive Group LASSO path for F^(0)
#
# Solve:
#
#   min_B  1/(2n) ||A_data - Theta B||_F^2
#
#          + lambda * sum_S w_S ||B_S||_2
#
# using warm-started proximal gradient.
#
# For every distinct active structural model:
#   - unpenalized OLS refit
#   - compute training RSS
#   - compute BIC
#
# BIC selects the final model.
#
# No oracle information is used.
# ================================================================

import time

cell8_start_time = time.perf_counter()


# --------------------------------------------------------------
# 1. Smooth-loss Lipschitz constant
#
# Gradient:
#
#   grad_B = Theta^T (Theta B - Y) / n
#
# Lipschitz constant:
#
#   L = ||Theta||_2^2 / n
# --------------------------------------------------------------

spectral_norm_theta = np.linalg.norm(
    Theta_train,
    ord=2
)

L_F0 = (
    spectral_norm_theta ** 2
    /
    N_TRAIN
)

STEP_F0 = 1.0 / L_F0


print(
    "Gradient Lipschitz constant =",
    f"{L_F0:.6f}"
)

print(
    "Proximal step size =",
    f"{STEP_F0:.6f}"
)


# --------------------------------------------------------------
# 2. Weighted group proximal operator
#
# Groups are disjoint and exhaustive.
# --------------------------------------------------------------

def adaptive_group_prox(
    B,
    threshold_lambda,
    group_weights,
):
    """
    Proximal operator for

        threshold_lambda *
        sum_S w_S ||B_S||_2
    """

    beta_flat = B.ravel(
        order="C"
    )

    output_flat = beta_flat.copy()

    for support, indices in structural_groups.items():

        group_vector = beta_flat[
            indices
        ]

        group_norm = np.linalg.norm(
            group_vector
        )

        threshold = (
            threshold_lambda
            *
            group_weights[
                support
            ]
        )

        if group_norm <= threshold:

            output_flat[
                indices
            ] = 0.0

        else:

            shrinkage = (
                1.0
                -
                threshold
                /
                group_norm
            )

            output_flat[
                indices
            ] *= shrinkage

    return output_flat.reshape(
        B.shape,
        order="C"
    )


# --------------------------------------------------------------
# 3. Proximal-gradient solver
#
# Updated numerical settings:
#
#   max_iter = 5000
#   tol      = 1e-8
#
# Penalized coefficients are used for support selection only;
# final coefficients are obtained by OLS refitting.
# --------------------------------------------------------------

def fit_adaptive_group_lasso(
    Theta,
    Y,
    lambda_value,
    group_weights,
    B_start=None,
    step_size=STEP_F0,
    max_iter=5000,
    tol=1e-8,
):
    """
    Warm-start compatible proximal-gradient solver.

    Returns
    -------
    B : ndarray
        Penalized coefficient matrix.

    info : dict
        Convergence diagnostics.
    """

    n_samples, n_features = Theta.shape
    n_outputs = Y.shape[1]

    if B_start is None:

        B = np.zeros(
            (
                n_features,
                n_outputs
            ),
            dtype=float
        )

    else:

        B = np.array(
            B_start,
            dtype=float,
            copy=True
        )


    relative_change = np.inf

    for iteration in range(
        1,
        max_iter + 1
    ):

        residual = (
            Theta @ B
            -
            Y
        )

        gradient = (
            Theta.T
            @ residual
            /
            n_samples
        )

        B_new = adaptive_group_prox(
            B - step_size * gradient,
            step_size * lambda_value,
            group_weights,
        )


        change = np.linalg.norm(
            B_new - B
        )

        reference = max(
            1.0,
            np.linalg.norm(B)
        )

        relative_change = (
            change
            /
            reference
        )

        B = B_new


        if relative_change < tol:
            break


    converged = (
        relative_change < tol
    )

    return B, {
        "iterations":
            iteration,

        "relative_change":
            float(relative_change),

        "converged":
            bool(converged),
    }


# --------------------------------------------------------------
# 4. Active structural support readout
# --------------------------------------------------------------

ACTIVE_GROUP_TOL = 1e-10


def active_supports_from_B(
    B,
    tol=ACTIVE_GROUP_TOL,
):

    beta_flat = B.ravel(
        order="C"
    )

    active = []

    for support, indices in structural_groups.items():

        group_norm = np.linalg.norm(
            beta_flat[
                indices
            ]
        )

        if group_norm > tol:

            active.append(
                support
            )

    return tuple(
        sorted(
            active,
            key=lambda s: (
                len(s),
                s
            )
        )
    )


# --------------------------------------------------------------
# 5. Precompute coefficient -> structural support lookup
#
# Avoid repeated pandas .loc calls inside every OLS refit.
# --------------------------------------------------------------

coefficient_support_lookup = (
    coefficient_ledger[
        "structural_support"
    ]
    .tolist()
)


# --------------------------------------------------------------
# 6. Unpenalized OLS refit on selected support model
#
# Group LASSO selects supports.
# OLS removes shrinkage bias.
# --------------------------------------------------------------

def refit_active_model(
    Theta,
    Y,
    active_supports,
):

    B_refit = np.zeros(
        (
            Theta.shape[1],
            Y.shape[1]
        ),
        dtype=float
    )

    active_set = set(
        active_supports
    )


    for output_index in range(
        Y.shape[1]
    ):

        active_features = []

        for feature_index in range(
            Theta.shape[1]
        ):

            flat_index = (
                feature_index * N
                +
                output_index
            )

            support = (
                coefficient_support_lookup[
                    flat_index
                ]
            )

            if support in active_set:

                active_features.append(
                    feature_index
                )


        if len(active_features) == 0:
            continue


        active_features = np.array(
            active_features,
            dtype=int
        )


        coefficients, *_ = np.linalg.lstsq(
            Theta[
                :,
                active_features
            ],
            Y[
                :,
                output_index
            ],
            rcond=None,
        )


        B_refit[
            active_features,
            output_index
        ] = coefficients


    return B_refit


# --------------------------------------------------------------
# 7. Model statistics and BIC
#
# All component observations are treated as scalar observations:
#
#     n_obs = N_TRAIN * N
# --------------------------------------------------------------

N_OBS_F0 = (
    N_TRAIN * N
)


def model_statistics(
    Theta,
    Y,
    B_refit,
    active_supports,
):

    prediction = (
        Theta
        @ B_refit
    )

    residual = (
        Y
        -
        prediction
    )

    rss = float(
        np.sum(
            residual ** 2
        )
    )


    n_parameters = int(
        sum(
            len(
                structural_groups[
                    support
                ]
            )

            for support in active_supports
        )
    )


    rss_safe = max(
        rss,
        np.finfo(float).tiny
    )


    bic = (
        N_OBS_F0
        *
        np.log(
            rss_safe
            /
            N_OBS_F0
        )
        +
        n_parameters
        *
        np.log(
            N_OBS_F0
        )
    )


    relative_error = float(
        np.linalg.norm(
            residual
        )
        /
        np.linalg.norm(
            Y
        )
    )


    return {
        "rss":
            rss,

        "bic":
            float(bic),

        "n_parameters":
            n_parameters,

        "relative_error":
            relative_error,
    }


# --------------------------------------------------------------
# 8. Run descending lambda path
#
# Warm starts make consecutive problems substantially cheaper.
# Progress is printed every five lambda values.
# --------------------------------------------------------------

path_rows_F0 = []

path_models_F0 = {}


B_warm = np.zeros(
    (
        N_FEATURES,
        N
    ),
    dtype=float
)


print()
print("Starting Adaptive Group-LASSO path...")
print()


for path_index, lambda_value in enumerate(
    lambda_path_F0
):

    lambda_start_time = time.perf_counter()


    B_penalized, solver_info = (
        fit_adaptive_group_lasso(
            Theta_train,
            A_train_data,
            lambda_value,
            adaptive_group_weights,
            B_start=B_warm,
        )
    )


    # Warm start for next lambda
    B_warm = B_penalized


    active_supports = (
        active_supports_from_B(
            B_penalized
        )
    )


    model_key = active_supports


    # ----------------------------------------------------------
    # Refit only when a new structural model appears
    # ----------------------------------------------------------

    if model_key not in path_models_F0:

        B_refit = refit_active_model(
            Theta_train,
            A_train_data,
            active_supports,
        )

        stats = model_statistics(
            Theta_train,
            A_train_data,
            B_refit,
            active_supports,
        )


        path_models_F0[
            model_key
        ] = {
            "B_refit":
                B_refit,

            "stats":
                stats,
        }

    else:

        stats = (
            path_models_F0[
                model_key
            ][
                "stats"
            ]
        )


    lambda_elapsed = (
        time.perf_counter()
        -
        lambda_start_time
    )


    path_rows_F0.append({
        "path_index":
            path_index,

        "lambda":
            float(
                lambda_value
            ),

        "n_groups":
            len(
                active_supports
            ),

        "n_parameters":
            stats[
                "n_parameters"
            ],

        "train_relative_error":
            stats[
                "relative_error"
            ],

        "rss":
            stats[
                "rss"
            ],

        "bic":
            stats[
                "bic"
            ],

        "solver_iterations":
            solver_info[
                "iterations"
            ],

        "solver_converged":
            solver_info[
                "converged"
            ],

        "solver_relative_change":
            solver_info[
                "relative_change"
            ],

        "lambda_runtime":
            float(
                lambda_elapsed
            ),

        "active_supports":
            active_supports,
    })


    # ----------------------------------------------------------
    # Progress monitor
    # ----------------------------------------------------------

    if (
        path_index % 5 == 0
        or
        path_index
        ==
        len(lambda_path_F0) - 1
    ):

        total_elapsed = (
            time.perf_counter()
            -
            cell8_start_time
        )

        print(
            f"[{path_index + 1:02d}"
            f"/{len(lambda_path_F0)}] "
            f"lambda={lambda_value:.3e} | "
            f"iter={solver_info['iterations']:4d} | "
            f"conv={solver_info['converged']} | "
            f"groups={len(active_supports):3d} | "
            f"lambda_time={lambda_elapsed:.2f}s | "
            f"elapsed={total_elapsed:.1f}s"
        )


    # Immediate warning for a non-converged point
    if not solver_info[
        "converged"
    ]:

        print(
            "  WARNING: lambda point did not "
            "converge within 5000 iterations:",
            f"{lambda_value:.6e}"
        )


path_ledger_F0 = pd.DataFrame(
    path_rows_F0
)


# --------------------------------------------------------------
# 9. Select model by minimum BIC
# --------------------------------------------------------------

best_row_index_F0 = (
    path_ledger_F0[
        "bic"
    ].idxmin()
)


best_row_F0 = (
    path_ledger_F0.loc[
        best_row_index_F0
    ]
)


selected_supports_F0 = (
    best_row_F0[
        "active_supports"
    ]
)


B0_hat_scaled = (
    path_models_F0[
        selected_supports_F0
    ][
        "B_refit"
    ]
)


# --------------------------------------------------------------
# 10. Held-out evaluation
# --------------------------------------------------------------

A_train_hat = (
    Theta_train
    @ B0_hat_scaled
)

A_test_hat = (
    Theta_test
    @ B0_hat_scaled
)


F0_train_relerr = (
    relative_field_error(
        A_train_hat,
        A_train_data,
    )
)

F0_test_relerr = (
    relative_field_error(
        A_test_hat,
        A_test_data,
    )
)


# --------------------------------------------------------------
# 11. Structural summary
# --------------------------------------------------------------

selected_count_by_size_F0 = Counter(
    len(support)
    for support in selected_supports_F0
)


cell8_elapsed = (
    time.perf_counter()
    -
    cell8_start_time
)


print()
print(
    "Adaptive Group LASSO F0 selection complete."
)

print()
print("Runtime:")
print(
    "  total =",
    f"{cell8_elapsed:.1f} s"
)

print()
print("Path:")
print(
    "  lambda points =",
    len(
        lambda_path_F0
    )
)

print(
    "  distinct structural models =",
    len(
        path_models_F0
    )
)

print(
    "  all solver runs converged =",
    bool(
        path_ledger_F0[
            "solver_converged"
        ].all()
    )
)

print(
    "  max iterations used =",
    int(
        path_ledger_F0[
            "solver_iterations"
        ].max()
    )
)


print()
print("BIC-selected model:")

print(
    "  lambda =",
    f"{best_row_F0['lambda']:.6e}"
)

print(
    "  active groups =",
    len(
        selected_supports_F0
    )
)

print(
    "  active coefficients =",
    int(
        best_row_F0[
            "n_parameters"
        ]
    )
)

print(
    "  groups by support size =",
    dict(
        sorted(
            selected_count_by_size_F0.items()
        )
    )
)


print()
print("Refitted field error:")

print(
    "  train =",
    f"{F0_train_relerr:.3e}"
)

print(
    "  test  =",
    f"{F0_test_relerr:.3e}"
)


print()
print("Selected supports:")

for support in selected_supports_F0:
    print(
        " ",
        support
    )


print()
print("Best BIC neighborhood:")

display(
    path_ledger_F0[
        [
            "path_index",
            "lambda",
            "n_groups",
            "n_parameters",
            "train_relative_error",
            "bic",
            "solver_iterations",
            "solver_converged",
            "lambda_runtime",
        ]
    ]
    .sort_values(
        "bic"
    )
    .head(12)
)

Gradient Lipschitz constant = 6.827258
Proximal step size = 0.146472

Starting Adaptive Group-LASSO path...

[01/80] lambda=1.315e-01 | iter=   1 | conv=True | groups=  0 | lambda_time=0.01s | elapsed=0.1s
[06/80] lambda=5.486e-02 | iter=  63 | conv=True | groups=  6 | lambda_time=0.32s | elapsed=1.5s
[11/80] lambda=2.288e-02 | iter= 171 | conv=True | groups=  8 | lambda_time=0.72s | elapsed=4.3s
[16/80] lambda=9.545e-03 | iter= 369 | conv=True | groups= 12 | lambda_time=1.45s | elapsed=9.7s
[21/80] lambda=3.981e-03 | iter= 613 | conv=True | groups= 20 | lambda_time=2.53s | elapsed=21.4s
[26/80] lambda=1.661e-03 | iter= 788 | conv=True | groups= 20 | lambda_time=2.43s | elapsed=34.2s
[31/80] lambda=6.927e-04 | iter= 863 | conv=True | groups= 20 | lambda_time=2.53s | elapsed=46.3s
[36/80] lambda=2.889e-04 | iter= 870 | conv=True | groups= 20 | lambda_time=5.00s | elapsed=63.5s
[41/80] lambda=1.205e-04 | iter= 838 | conv=True | groups= 20 | lambda_time=5.43s | elapsed=84.2s
[46/80] lambd

,path_index,lambda,n_groups,n_parameters,train_relative_error,bic,solver_iterations,solver_converged,lambda_runtime
78,78,1.566629e-07,25,228,7.492688e-10,-1.056348e+06,349,True,0.924672
73,73,3.755891e-07,25,228,7.492688e-10,-1.056348e+06,390,True,1.070444
75,75,2.647368e-07,25,228,7.492688e-10,-1.056348e+06,393,True,1.087188
74,74,3.153288e-07,25,228,7.492688e-10,-1.056348e+06,408,True,1.230203
79,79,1.315276e-07,25,228,7.492688e-10,-1.056348e+06,335,True,1.059438
77,77,1.866017e-07,25,228,7.492688e-10,-1.056348e+06,364,True,0.957405
76,76,2.222618e-07,25,228,7.492688e-10,-1.056348e+06,379,True,1.063547
63,63,2.158772e-06,22,192,2.046906e-03,-3.453268e+05,530,True,1.650909
64,64,1.812414e-06,22,192,2.046906e-03,-3.453268e+05,514,True,1.442356
65,65,1.521626e-06,22,192,2.046906e-03,-3.453268e+05,499,True,1.388616


In [11]:
# ================================================================
# Cell 9 — Finite-flow correction from learned F^(0)
#
# Forward endpoint expansion:
#
#   Y_F(x, epsilon)
#     = F0(x)
#       + epsilon [
#           F1(x)
#           + 1/2 D F0(x) F0(x)
#         ]
#       + O(epsilon^2)
#
# Cell 5 estimated:
#
#   A_data  ~ F0
#   G1_data ~ F1 + Q[F0]
#
# where
#
#   Q[F0] = 1/2 D F0 F0.
#
# Here Q is computed entirely from the learned F0 model.
#
# No oracle information is used.
# ================================================================


# --------------------------------------------------------------
# 1. Convert normalized-library coefficients back to raw
#    monomial coefficients
#
# Theta_scaled = Theta_raw / feature_scale
#
# Therefore:
#
# Theta_scaled @ B_scaled
#     = Theta_raw @ B_raw
#
# with
#
# B_raw = B_scaled / feature_scale
# --------------------------------------------------------------

B0_hat_raw = (
    B0_hat_scaled
    /
    feature_scale[:, None]
)


assert B0_hat_raw.shape == (
    N_FEATURES,
    N
)


# --------------------------------------------------------------
# 2. Evaluate learned polynomial vector field
# --------------------------------------------------------------

def evaluate_polynomial_field_raw(
    X,
    B_raw,
):
    """
    Evaluate a polynomial vector field represented in the
    unscaled monomial basis.

    Parameters
    ----------
    X : ndarray, shape (n_samples, N)

    B_raw : ndarray, shape (N_FEATURES, N)

    Returns
    -------
    F : ndarray, shape (n_samples, N)
    """

    Theta_raw = evaluate_library(
        X
    )

    return (
        Theta_raw
        @ B_raw
    )


# --------------------------------------------------------------
# 3. Evaluate Jacobian of learned polynomial vector field
#
# J[n, i, j]
#     = d F_i / d x_j
# --------------------------------------------------------------

def evaluate_polynomial_jacobian_raw(
    X,
    B_raw,
):
    """
    Analytic Jacobian of polynomial vector field.

    Returns
    -------
    J : ndarray
        shape (n_samples, N, N)

        J[:, i, j] =
            partial F_i / partial x_j
    """

    n_samples = X.shape[0]

    J = np.zeros(
        (
            n_samples,
            N,
            N
        ),
        dtype=float
    )


    for feature_index, exponent in enumerate(
        monomial_exponents
    ):

        for variable_index in range(N):

            power = exponent[
                variable_index
            ]

            if power == 0:
                continue


            # derivative exponent:
            #
            # d/dx_j x^alpha
            #
            #   = alpha_j
            #     x^(alpha - e_j)

            derivative_values = np.ones(
                n_samples,
                dtype=float
            )

            for k, exponent_k in enumerate(
                exponent
            ):

                derivative_power = (
                    exponent_k
                    -
                    (1 if k == variable_index else 0)
                )

                if derivative_power:

                    derivative_values *= (
                        X[:, k]
                        ** derivative_power
                    )


            derivative_values *= power


            # B_raw[feature_index, output_index]
            #
            # contributes to every output component.

            J[
                :,
                :,
                variable_index
            ] += (
                derivative_values[:, None]
                *
                B_raw[
                    feature_index,
                    :
                ][None, :]
            )


    return J


# --------------------------------------------------------------
# 4. Q[F0] = 1/2 D F0 F0
# --------------------------------------------------------------

def finite_flow_correction(
    X,
    B_raw,
):
    """
    Compute

        Q[F] = 1/2 DF(x) F(x)

    directly from learned polynomial coefficients.
    """

    F = evaluate_polynomial_field_raw(
        X,
        B_raw
    )

    J = evaluate_polynomial_jacobian_raw(
        X,
        B_raw
    )


    Q = 0.5 * np.einsum(
        "nij,nj->ni",
        J,
        F
    )


    return F, J, Q


(
    F0_train_from_raw,
    J0_train_hat,
    Q_train_hat,
) = finite_flow_correction(
    learner_data["X0_train"],
    B0_hat_raw,
)


(
    F0_test_from_raw,
    J0_test_hat,
    Q_test_hat,
) = finite_flow_correction(
    learner_data["X0_test"],
    B0_hat_raw,
)


# --------------------------------------------------------------
# 5. Internal representation check
#
# Raw and normalized representations of learned F0 must agree.
# --------------------------------------------------------------

representation_error_train = (
    relative_field_error(
        F0_train_from_raw,
        A_train_hat,
    )
)

representation_error_test = (
    relative_field_error(
        F0_test_from_raw,
        A_test_hat,
    )
)


assert representation_error_train < 1e-12
assert representation_error_test < 1e-12


# --------------------------------------------------------------
# 6. Construct chronology pseudo-target
#
# F1_target =
#
#     G1_data
#     -
#     Q[F0_hat]
# --------------------------------------------------------------

F1_train_target = (
    G1_train_data
    -
    Q_train_hat
)

F1_test_target = (
    G1_test_data
    -
    Q_test_hat
)


# --------------------------------------------------------------
# 7. Data-only diagnostics
# --------------------------------------------------------------

def field_rms(X):
    return float(
        np.sqrt(
            np.mean(
                X ** 2
            )
        )
    )


G1_rms = field_rms(
    G1_train_data
)

Q_rms = field_rms(
    Q_train_hat
)

F1_target_rms = field_rms(
    F1_train_target
)


Q_fraction_of_G1 = (
    Q_rms
    /
    G1_rms
)


# Conservation should persist.
Q_conservation = np.max(
    np.abs(
        Q_train_hat.sum(
            axis=1
        )
    )
)

F1_target_conservation = np.max(
    np.abs(
        F1_train_target.sum(
            axis=1
        )
    )
)


print(
    "Learned finite-flow correction complete."
)

print()
print("Representation check:")
print(
    "  train raw/scaled relative error =",
    f"{representation_error_train:.3e}"
)
print(
    "  test raw/scaled relative error  =",
    f"{representation_error_test:.3e}"
)

print()
print("Field magnitudes:")
print(
    "  RMS(G1_data)   =",
    f"{G1_rms:.8f}"
)
print(
    "  RMS(Q[F0_hat]) =",
    f"{Q_rms:.8f}"
)
print(
    "  RMS(F1_target) =",
    f"{F1_target_rms:.8f}"
)
print(
    "  RMS(Q) / RMS(G1) =",
    f"{Q_fraction_of_G1:.6f}"
)

print()
print("Conservation:")
print(
    "  max |sum_i Q_i| =",
    f"{Q_conservation:.3e}"
)
print(
    "  max |sum_i F1_target_i| =",
    f"{F1_target_conservation:.3e}"
)

Learned finite-flow correction complete.

Representation check:
  train raw/scaled relative error = 1.246e-16
  test raw/scaled relative error  = 1.240e-16

Field magnitudes:
  RMS(G1_data)   = 0.29990084
  RMS(Q[F0_hat]) = 0.29930602
  RMS(F1_target) = 0.01711346
  RMS(Q) / RMS(G1) = 0.998017

Conservation:
  max |sum_i Q_i| = 1.060e-08
  max |sum_i F1_target_i| = 1.047e-08


In [12]:
# ================================================================
# Cell 10 — Adaptive Group-LASSO initialization for F^(1)
#
# Target:
#
#     F1_target
#       = G1_data - Q[F0_hat]
#
# Steps:
#   1. OLS initial estimator
#   2. structural group norms
#   3. adaptive group weights
#   4. weighted lambda_max
#   5. frozen lambda path
#
# Same generic polynomial library and same structural groups
# as for F^(0).
#
# No oracle information is used.
# ================================================================


# --------------------------------------------------------------
# 1. Initial OLS estimator
# --------------------------------------------------------------

B1_init_scaled, *_ = np.linalg.lstsq(
    Theta_train,
    F1_train_target,
    rcond=None,
)


assert B1_init_scaled.shape == (
    N_FEATURES,
    N,
)


F1_train_init = (
    Theta_train
    @ B1_init_scaled
)

F1_test_init = (
    Theta_test
    @ B1_init_scaled
)


F1_init_train_relerr = relative_field_error(
    F1_train_init,
    F1_train_target,
)

F1_init_test_relerr = relative_field_error(
    F1_test_init,
    F1_test_target,
)


# --------------------------------------------------------------
# 2. Initial structural group norms
# --------------------------------------------------------------

beta1_init_flat = (
    B1_init_scaled
    .ravel(order="C")
)


initial_group_norms_F1 = {}

for support, indices in structural_groups.items():

    initial_group_norms_F1[
        support
    ] = float(
        np.linalg.norm(
            beta1_init_flat[
                indices
            ]
        )
    )


group_norm_values_F1 = np.array(
    list(
        initial_group_norms_F1.values()
    )
)


# --------------------------------------------------------------
# 3. Adaptive weights
#
# Same rule as F0:
#
#   w_S =
#       sqrt(|g_S|)
#       /
#       (||beta_init,S||_2 + delta)^gamma
#
# gamma remains frozen at 1.
# --------------------------------------------------------------

ADAPTIVE_GAMMA_F1 = 1.0

ADAPTIVE_DELTA_F1 = max(
    1e-12,
    1e-8
    *
    group_norm_values_F1.max(),
)


adaptive_group_weights_F1 = {}

for support, indices in structural_groups.items():

    norm_init = (
        initial_group_norms_F1[
            support
        ]
    )

    adaptive_group_weights_F1[
        support
    ] = (
        np.sqrt(
            len(indices)
        )
        /
        (
            norm_init
            +
            ADAPTIVE_DELTA_F1
        )
        ** ADAPTIVE_GAMMA_F1
    )


weight_values_F1 = np.array(
    list(
        adaptive_group_weights_F1.values()
    )
)


# --------------------------------------------------------------
# 4. Weighted lambda_max
#
# Same objective convention:
#
#   1/(2n) ||F1_target - Theta B||_F^2
#
#   + lambda sum_S w_S ||B_S||_2
# --------------------------------------------------------------

gradient_at_zero_F1 = (
    -Theta_train.T
    @ F1_train_target
    /
    N_TRAIN
)


gradient_zero_flat_F1 = (
    gradient_at_zero_F1
    .ravel(order="C")
)


lambda_candidates_F1 = []

for support, indices in structural_groups.items():

    grad_group_norm = np.linalg.norm(
        gradient_zero_flat_F1[
            indices
        ]
    )

    weight = (
        adaptive_group_weights_F1[
            support
        ]
    )

    lambda_candidates_F1.append(
        grad_group_norm
        /
        weight
    )


LAMBDA_MAX_F1 = float(
    np.max(
        lambda_candidates_F1
    )
)


# --------------------------------------------------------------
# 5. Frozen regularization path
# --------------------------------------------------------------

N_LAMBDAS_F1 = 80
LAMBDA_MIN_RATIO_F1 = 1e-6


lambda_path_F1 = (
    LAMBDA_MAX_F1
    *
    np.geomspace(
        1.0,
        LAMBDA_MIN_RATIO_F1,
        N_LAMBDAS_F1,
    )
)


assert np.all(
    np.diff(
        lambda_path_F1
    ) < 0
)


# --------------------------------------------------------------
# 6. Diagnostic ledger
# --------------------------------------------------------------

adaptive_rows_F1 = []

for support in sorted(
    structural_groups,
    key=lambda s: (
        len(s),
        s
    )
):

    adaptive_rows_F1.append({
        "support":
            support,

        "support_size":
            len(support),

        "group_size":
            len(
                structural_groups[
                    support
                ]
            ),

        "initial_norm":
            initial_group_norms_F1[
                support
            ],

        "adaptive_weight":
            adaptive_group_weights_F1[
                support
            ],
    })


adaptive_weight_ledger_F1 = pd.DataFrame(
    adaptive_rows_F1
)


# --------------------------------------------------------------
# 7. Summary
# --------------------------------------------------------------

print(
    "Adaptive Group-LASSO F1 initialization complete."
)

print()
print("Initial OLS fit:")
print(
    "  train relative field error =",
    f"{F1_init_train_relerr:.3e}"
)
print(
    "  test relative field error  =",
    f"{F1_init_test_relerr:.3e}"
)

print()
print("Initial group norms:")
print(
    "  min =",
    f"{group_norm_values_F1.min():.3e}"
)
print(
    "  median =",
    f"{np.median(group_norm_values_F1):.3e}"
)
print(
    "  max =",
    f"{group_norm_values_F1.max():.3e}"
)

print()
print("Adaptive weights:")
print(
    "  gamma =",
    ADAPTIVE_GAMMA_F1
)
print(
    "  delta =",
    f"{ADAPTIVE_DELTA_F1:.3e}"
)
print(
    "  min =",
    f"{weight_values_F1.min():.3e}"
)
print(
    "  median =",
    f"{np.median(weight_values_F1):.3e}"
)
print(
    "  max =",
    f"{weight_values_F1.max():.3e}"
)

print()
print("Regularization path:")
print(
    "  lambda_max =",
    f"{LAMBDA_MAX_F1:.6e}"
)
print(
    "  lambda_min =",
    f"{lambda_path_F1[-1]:.6e}"
)
print(
    "  number of lambdas =",
    N_LAMBDAS_F1
)

print()
print("Largest initial structural groups:")
display(
    adaptive_weight_ledger_F1
    .sort_values(
        "initial_norm",
        ascending=False
    )
    .head(30)
)

Adaptive Group-LASSO F1 initialization complete.

Initial OLS fit:
  train relative field error = 1.472e-06
  test relative field error  = 1.548e-06

Initial group norms:
  min = 2.296e-10
  median = 1.350e-08
  max = 1.590e-02

Adaptive weights:
  gamma = 1.0
  delta = 1.590e-10
  min = 2.179e+02
  median = 2.339e+08
  max = 5.146e+09

Regularization path:
  lambda_max = 1.021031e-04
  lambda_min = 1.021031e-10
  number of lambdas = 80

Largest initial structural groups:


,support,support_size,group_size,initial_norm,adaptive_weight
9,"(1, 3)",2,12,0.015901,217.858883
35,"(7, 8)",2,12,0.015691,220.769014
8,"(1, 2)",2,12,0.014724,235.273042
34,"(6, 8)",2,12,0.011882,291.539293
25,"(3, 8)",2,12,0.011258,307.711949
14,"(1, 8)",2,12,0.011172,310.062569
20,"(2, 8)",2,12,0.010970,315.776384
16,"(2, 4)",2,12,0.010964,315.947501
32,"(5, 8)",2,12,0.010958,316.117179
28,"(4, 7)",2,12,0.010941,316.624162


In [13]:
# ================================================================
# Cell 11 — Adaptive Group LASSO path for F^(1)
#
# Target:
#
#     F1_target = G1_data - Q[F0_hat]
#
# Reuse:
#   - proximal solver from Cell 8
#   - structural groups from Cell 6
#   - OLS refit from Cell 8
#
# Model selection:
#   refitted BIC
#
# No oracle information is used.
# ================================================================

import time


# --------------------------------------------------------------
# 1. Generic model statistics
# --------------------------------------------------------------

def model_statistics_generic(
    Theta,
    Y,
    B_refit,
    active_supports,
):
    prediction = Theta @ B_refit

    residual = Y - prediction

    rss = float(
        np.sum(
            residual ** 2
        )
    )

    n_parameters = int(
        sum(
            len(
                structural_groups[
                    support
                ]
            )
            for support in active_supports
        )
    )

    n_obs = (
        Y.shape[0]
        *
        Y.shape[1]
    )

    rss_safe = max(
        rss,
        np.finfo(float).tiny
    )

    bic = (
        n_obs
        *
        np.log(
            rss_safe / n_obs
        )
        +
        n_parameters
        *
        np.log(
            n_obs
        )
    )

    relative_error = float(
        np.linalg.norm(
            residual
        )
        /
        np.linalg.norm(
            Y
        )
    )

    return {
        "rss": rss,
        "bic": float(bic),
        "n_parameters": n_parameters,
        "relative_error": relative_error,
    }


# --------------------------------------------------------------
# 2. Generic Adaptive Group-LASSO path runner
# --------------------------------------------------------------

def run_adaptive_gl_path(
    Theta,
    Y,
    lambda_path,
    group_weights,
    label="model",
    progress_every=5,
):
    """
    Run descending warm-started Adaptive Group-LASSO path,
    refit every distinct support model by OLS, and compute BIC.
    """

    start_time = time.perf_counter()

    path_rows = []
    path_models = {}

    B_warm = np.zeros(
        (
            Theta.shape[1],
            Y.shape[1]
        ),
        dtype=float
    )

    print(
        f"Starting Adaptive Group-LASSO path for {label}..."
    )
    print()

    for path_index, lambda_value in enumerate(
        lambda_path
    ):

        lambda_start = time.perf_counter()

        B_penalized, solver_info = (
            fit_adaptive_group_lasso(
                Theta,
                Y,
                lambda_value,
                group_weights,
                B_start=B_warm,
                step_size=STEP_F0,   # same Theta => same Lipschitz constant
                max_iter=5000,
                tol=1e-8,
            )
        )

        B_warm = B_penalized

        active_supports = (
            active_supports_from_B(
                B_penalized
            )
        )

        model_key = active_supports


        # ------------------------------------------------------
        # OLS refit only once per distinct structural model
        # ------------------------------------------------------

        if model_key not in path_models:

            B_refit = refit_active_model(
                Theta,
                Y,
                active_supports,
            )

            stats = model_statistics_generic(
                Theta,
                Y,
                B_refit,
                active_supports,
            )

            path_models[
                model_key
            ] = {
                "B_refit": B_refit,
                "stats": stats,
            }

        else:

            stats = (
                path_models[
                    model_key
                ]["stats"]
            )


        lambda_runtime = (
            time.perf_counter()
            -
            lambda_start
        )


        path_rows.append({
            "path_index":
                path_index,

            "lambda":
                float(lambda_value),

            "n_groups":
                len(active_supports),

            "n_parameters":
                stats[
                    "n_parameters"
                ],

            "train_relative_error":
                stats[
                    "relative_error"
                ],

            "rss":
                stats[
                    "rss"
                ],

            "bic":
                stats[
                    "bic"
                ],

            "solver_iterations":
                solver_info[
                    "iterations"
                ],

            "solver_converged":
                solver_info[
                    "converged"
                ],

            "solver_relative_change":
                solver_info[
                    "relative_change"
                ],

            "lambda_runtime":
                float(
                    lambda_runtime
                ),

            "active_supports":
                active_supports,
        })


        # ------------------------------------------------------
        # Progress
        # ------------------------------------------------------

        if (
            path_index % progress_every == 0
            or
            path_index == len(lambda_path) - 1
        ):

            elapsed = (
                time.perf_counter()
                -
                start_time
            )

            print(
                f"[{path_index + 1:02d}/{len(lambda_path)}] "
                f"lambda={lambda_value:.3e} | "
                f"iter={solver_info['iterations']:4d} | "
                f"conv={solver_info['converged']} | "
                f"groups={len(active_supports):3d} | "
                f"lambda_time={lambda_runtime:.2f}s | "
                f"elapsed={elapsed:.1f}s"
            )


        if not solver_info[
            "converged"
        ]:

            print(
                "  WARNING: non-converged lambda:",
                f"{lambda_value:.6e}"
            )


    ledger = pd.DataFrame(
        path_rows
    )

    elapsed = (
        time.perf_counter()
        -
        start_time
    )

    return (
        ledger,
        path_models,
        elapsed,
    )


# --------------------------------------------------------------
# 3. Run F1 path
# --------------------------------------------------------------

(
    path_ledger_F1,
    path_models_F1,
    cell11_elapsed,
) = run_adaptive_gl_path(
    Theta_train,
    F1_train_target,
    lambda_path_F1,
    adaptive_group_weights_F1,
    label="F1",
)


# --------------------------------------------------------------
# 4. BIC model selection
# --------------------------------------------------------------

best_row_index_F1 = (
    path_ledger_F1[
        "bic"
    ].idxmin()
)

best_row_F1 = (
    path_ledger_F1.loc[
        best_row_index_F1
    ]
)

selected_supports_F1 = (
    best_row_F1[
        "active_supports"
    ]
)

B1_hat_scaled = (
    path_models_F1[
        selected_supports_F1
    ][
        "B_refit"
    ]
)


# --------------------------------------------------------------
# 5. Held-out field evaluation
# --------------------------------------------------------------

F1_train_hat = (
    Theta_train
    @ B1_hat_scaled
)

F1_test_hat = (
    Theta_test
    @ B1_hat_scaled
)


F1_train_relerr = (
    relative_field_error(
        F1_train_hat,
        F1_train_target,
    )
)

F1_test_relerr = (
    relative_field_error(
        F1_test_hat,
        F1_test_target,
    )
)


# --------------------------------------------------------------
# 6. Structural summary
# --------------------------------------------------------------

selected_count_by_size_F1 = Counter(
    len(support)
    for support in selected_supports_F1
)


print()
print(
    "Adaptive Group LASSO F1 selection complete."
)

print()
print("Runtime:")
print(
    "  total =",
    f"{cell11_elapsed:.1f} s"
)

print()
print("Path:")
print(
    "  lambda points =",
    len(
        lambda_path_F1
    )
)

print(
    "  distinct structural models =",
    len(
        path_models_F1
    )
)

print(
    "  all solver runs converged =",
    bool(
        path_ledger_F1[
            "solver_converged"
        ].all()
    )
)

print(
    "  max iterations used =",
    int(
        path_ledger_F1[
            "solver_iterations"
        ].max()
    )
)


print()
print("BIC-selected model:")

print(
    "  lambda =",
    f"{best_row_F1['lambda']:.6e}"
)

print(
    "  active groups =",
    len(
        selected_supports_F1
    )
)

print(
    "  active coefficients =",
    int(
        best_row_F1[
            "n_parameters"
        ]
    )
)

print(
    "  groups by support size =",
    dict(
        sorted(
            selected_count_by_size_F1.items()
        )
    )
)


print()
print("Refitted field error:")

print(
    "  train =",
    f"{F1_train_relerr:.3e}"
)

print(
    "  test  =",
    f"{F1_test_relerr:.3e}"
)


print()
print("Selected supports:")

for support in selected_supports_F1:
    print(
        " ",
        support
    )


print()
print("Best BIC neighborhood:")

display(
    path_ledger_F1[
        [
            "path_index",
            "lambda",
            "n_groups",
            "n_parameters",
            "train_relative_error",
            "bic",
            "solver_iterations",
            "solver_converged",
            "lambda_runtime",
        ]
    ]
    .sort_values(
        "bic"
    )
    .head(15)
)

Starting Adaptive Group-LASSO path for F1...

[01/80] lambda=1.021e-04 | iter=   1 | conv=True | groups=  0 | lambda_time=0.01s | elapsed=0.0s
[06/80] lambda=4.259e-05 | iter=  47 | conv=True | groups= 11 | lambda_time=0.26s | elapsed=1.0s
[11/80] lambda=1.776e-05 | iter= 117 | conv=True | groups= 18 | lambda_time=0.61s | elapsed=3.3s
[16/80] lambda=7.410e-06 | iter= 249 | conv=True | groups= 24 | lambda_time=0.89s | elapsed=7.6s
[21/80] lambda=3.091e-06 | iter= 405 | conv=True | groups= 29 | lambda_time=1.24s | elapsed=12.7s
[26/80] lambda=1.289e-06 | iter= 522 | conv=True | groups= 33 | lambda_time=1.31s | elapsed=19.2s
[31/80] lambda=5.377e-07 | iter= 583 | conv=True | groups= 37 | lambda_time=1.60s | elapsed=27.8s
[36/80] lambda=2.243e-07 | iter= 579 | conv=True | groups= 37 | lambda_time=1.83s | elapsed=36.4s
[41/80] lambda=9.355e-08 | iter= 550 | conv=True | groups= 37 | lambda_time=1.40s | elapsed=45.0s
[46/80] lambda=3.902e-08 | iter= 491 | conv=True | groups= 37 | lambda_time=

,path_index,lambda,n_groups,n_parameters,train_relative_error,bic,solver_iterations,solver_converged,lambda_runtime
79,79,1.021031e-10,55,610,0.000441,-559989.781605,86,True,0.506665
78,78,1.216154e-10,54,606,0.000533,-550933.778468,93,True,0.389635
76,76,1.725390e-10,53,602,0.000625,-543318.272188,105,True,0.554791
77,77,1.448565e-10,53,602,0.000625,-543318.272188,100,True,0.314247
75,75,2.055117e-10,52,590,0.000809,-531039.102708,107,True,0.434111
72,72,3.472839e-10,51,578,0.000980,-521984.976546,128,True,0.488351
73,73,2.915650e-10,51,578,0.000980,-521984.976546,125,True,0.350960
74,74,2.447857e-10,51,578,0.000980,-521984.976546,116,True,0.374862
71,71,4.136510e-10,49,570,0.001254,-510204.570256,119,True,0.487279
68,68,6.990080e-10,47,546,0.001694,-496024.852741,151,True,0.443490


In [14]:
# ================================================================
# Cell 11b — Extended F1 regularization path
#
# Diagnostic motivation:
#
# The previous BIC minimum occurred at the smallest available
# lambda. Therefore the regularization path was truncated before
# reaching an interior BIC optimum.
#
# Extend:
#
#     lambda_max
#       -> 1e-10 * lambda_max
#
# No oracle information is used.
# ================================================================


N_LAMBDAS_F1_EXT = 120

LAMBDA_MIN_RATIO_F1_EXT = 1e-10


lambda_path_F1_ext = (
    LAMBDA_MAX_F1
    *
    np.geomspace(
        1.0,
        LAMBDA_MIN_RATIO_F1_EXT,
        N_LAMBDAS_F1_EXT,
    )
)


print(
    "Extended F1 lambda path:"
)

print(
    "  lambda_max =",
    f"{lambda_path_F1_ext[0]:.6e}"
)

print(
    "  lambda_min =",
    f"{lambda_path_F1_ext[-1]:.6e}"
)

print(
    "  points =",
    len(lambda_path_F1_ext)
)


# --------------------------------------------------------------
# Run extended path
# --------------------------------------------------------------

(
    path_ledger_F1_ext,
    path_models_F1_ext,
    cell11b_elapsed,
) = run_adaptive_gl_path(
    Theta_train,
    F1_train_target,
    lambda_path_F1_ext,
    adaptive_group_weights_F1,
    label="F1 extended",
    progress_every=5,
)


# --------------------------------------------------------------
# BIC selection
# --------------------------------------------------------------

best_row_index_F1_ext = (
    path_ledger_F1_ext[
        "bic"
    ].idxmin()
)


best_row_F1_ext = (
    path_ledger_F1_ext.loc[
        best_row_index_F1_ext
    ]
)


selected_supports_F1_ext = (
    best_row_F1_ext[
        "active_supports"
    ]
)


B1_hat_scaled_ext = (
    path_models_F1_ext[
        selected_supports_F1_ext
    ][
        "B_refit"
    ]
)


# --------------------------------------------------------------
# Held-out field evaluation
# --------------------------------------------------------------

F1_train_hat_ext = (
    Theta_train
    @ B1_hat_scaled_ext
)

F1_test_hat_ext = (
    Theta_test
    @ B1_hat_scaled_ext
)


F1_train_relerr_ext = (
    relative_field_error(
        F1_train_hat_ext,
        F1_train_target,
    )
)

F1_test_relerr_ext = (
    relative_field_error(
        F1_test_hat_ext,
        F1_test_target,
    )
)


selected_count_by_size_F1_ext = Counter(
    len(support)
    for support in selected_supports_F1_ext
)


# --------------------------------------------------------------
# Is the optimum interior?
# --------------------------------------------------------------

best_path_index_F1_ext = int(
    best_row_F1_ext[
        "path_index"
    ]
)


bic_optimum_is_boundary = (
    best_path_index_F1_ext
    ==
    len(lambda_path_F1_ext) - 1
)


# --------------------------------------------------------------
# Summary
# --------------------------------------------------------------

print()
print(
    "Extended Adaptive Group LASSO F1 selection complete."
)

print()
print("Runtime:")

print(
    "  total =",
    f"{cell11b_elapsed:.1f} s"
)


print()
print("Path:")

print(
    "  lambda points =",
    len(
        lambda_path_F1_ext
    )
)

print(
    "  distinct structural models =",
    len(
        path_models_F1_ext
    )
)

print(
    "  all solver runs converged =",
    bool(
        path_ledger_F1_ext[
            "solver_converged"
        ].all()
    )
)

print(
    "  max iterations used =",
    int(
        path_ledger_F1_ext[
            "solver_iterations"
        ].max()
    )
)


print()
print("BIC-selected model:")

print(
    "  path index =",
    best_path_index_F1_ext,
    "/",
    len(lambda_path_F1_ext) - 1
)

print(
    "  lambda =",
    f"{best_row_F1_ext['lambda']:.6e}"
)

print(
    "  optimum at lower boundary =",
    bic_optimum_is_boundary
)

print(
    "  active groups =",
    len(
        selected_supports_F1_ext
    )
)

print(
    "  active coefficients =",
    int(
        best_row_F1_ext[
            "n_parameters"
        ]
    )
)

print(
    "  groups by support size =",
    dict(
        sorted(
            selected_count_by_size_F1_ext.items()
        )
    )
)


print()
print("Refitted field error:")

print(
    "  train =",
    f"{F1_train_relerr_ext:.3e}"
)

print(
    "  test  =",
    f"{F1_test_relerr_ext:.3e}"
)


print()
print("Selected supports:")

for support in selected_supports_F1_ext:

    print(
        " ",
        support
    )


print()
print("Best BIC neighborhood:")

display(
    path_ledger_F1_ext[
        [
            "path_index",
            "lambda",
            "n_groups",
            "n_parameters",
            "train_relative_error",
            "bic",
            "solver_iterations",
            "solver_converged",
            "lambda_runtime",
        ]
    ]
    .sort_values(
        "bic"
    )
    .head(20)
)

Extended F1 lambda path:
  lambda_max = 1.021031e-04
  lambda_min = 1.021031e-14
  points = 120
Starting Adaptive Group-LASSO path for F1 extended...

[01/120] lambda=1.021e-04 | iter=   1 | conv=True | groups=  0 | lambda_time=0.01s | elapsed=0.0s
[06/120] lambda=3.880e-05 | iter=  51 | conv=True | groups= 12 | lambda_time=0.35s | elapsed=1.0s
[11/120] lambda=1.475e-05 | iter= 141 | conv=True | groups= 20 | lambda_time=0.58s | elapsed=3.2s
[16/120] lambda=5.604e-06 | iter= 305 | conv=True | groups= 25 | lambda_time=0.65s | elapsed=7.0s
[21/120] lambda=2.130e-06 | iter= 464 | conv=True | groups= 33 | lambda_time=1.48s | elapsed=13.2s
[26/120] lambda=8.095e-07 | iter= 574 | conv=True | groups= 36 | lambda_time=1.75s | elapsed=21.5s
[31/120] lambda=3.076e-07 | iter= 594 | conv=True | groups= 37 | lambda_time=1.76s | elapsed=30.6s
[36/120] lambda=1.169e-07 | iter= 571 | conv=True | groups= 37 | lambda_time=1.70s | elapsed=40.4s
[41/120] lambda=4.443e-08 | iter= 512 | conv=True | groups= 3

,path_index,lambda,n_groups,n_parameters,train_relative_error,bic,solver_iterations,solver_converged,lambda_runtime
119,119,1.021031e-14,59,632,0.000005,-778832.604313,2,True,0.009228
118,118,1.239004e-14,59,632,0.000005,-778832.604313,2,True,0.010048
117,117,1.503510e-14,59,632,0.000005,-778832.604313,3,True,0.015415
116,116,1.824484e-14,59,632,0.000005,-778832.604313,3,True,0.012676
115,115,2.213979e-14,59,632,0.000005,-778832.604313,4,True,0.016612
111,111,4.800738e-14,59,632,0.000005,-778832.604313,7,True,0.034842
110,110,5.825613e-14,59,632,0.000005,-778832.604313,7,True,0.029875
109,109,7.069281e-14,59,632,0.000005,-778832.604313,7,True,0.156547
113,113,3.260174e-14,59,632,0.000005,-778832.604313,5,True,0.020032
114,114,2.686626e-14,59,632,0.000005,-778832.604313,4,True,0.016997


In [15]:
# ================================================================
# Cell 12 — ORACLE VALIDATION
#
# IMPORTANT:
#
# This is the first cell in the inference notebook allowed to use
# the microscopic temporal generators.
#
# Everything above this cell is learner-only.
#
# Validate:
#
#   1. exact oracle F0 and F1
#   2. oracle structural supports
#   3. TP / FP / FN of learned supports
#   4. target-vs-oracle error
#   5. learned-vs-oracle generator error
#   6. coefficient-space error
# ================================================================


# --------------------------------------------------------------
# 1. Lie bracket
#
# Frozen convention:
#
#     [X,Y] = DY X - DX Y
#
# Chronology:
#
#     G1 -> G2 -> ... -> G6
#
# gives
#
#     F1 = (1/72) sum_{r<s} [F_r,F_s]
# --------------------------------------------------------------

def lie_bracket_symbolic(
    X,
    Y,
):
    JX = X.jacobian(
        x_symbols
    )

    JY = Y.jacobian(
        x_symbols
    )

    bracket = (
        JY * X
        -
        JX * Y
    )

    return sp.Matrix([
        sp.expand(component)
        for component in bracket
    ])


# --------------------------------------------------------------
# 2. Exact oracle F0
# --------------------------------------------------------------

F0_oracle_symbolic = sum(
    snapshot_fields_symbolic,
    sp.zeros(N, 1)
) / sp.Integer(
    n_snapshots
)


F0_oracle_symbolic = sp.Matrix([
    sp.expand(component)
    for component in F0_oracle_symbolic
])


# --------------------------------------------------------------
# 3. Exact oracle F1
# --------------------------------------------------------------

F1_oracle_symbolic = sp.zeros(
    N,
    1
)


for r in range(
    n_snapshots
):

    for s in range(
        r + 1,
        n_snapshots
    ):

        F1_oracle_symbolic += (
            lie_bracket_symbolic(
                snapshot_fields_symbolic[r],
                snapshot_fields_symbolic[s],
            )
        )


F1_oracle_symbolic = (
    F1_oracle_symbolic
    /
    sp.Integer(72)
)


F1_oracle_symbolic = sp.Matrix([
    sp.expand(component)
    for component in F1_oracle_symbolic
])


# --------------------------------------------------------------
# 4. Convert exact symbolic oracle into the same raw
#    polynomial coefficient basis used by the learner
# --------------------------------------------------------------

exponent_to_feature = {
    exponent:
        feature_index

    for feature_index, exponent
    in enumerate(
        monomial_exponents
    )
}


def symbolic_field_to_raw_coefficients(
    field_symbolic
):
    """
    Convert exact symbolic vector field into

        B_raw[feature, output]

    using the learner's generic monomial basis.
    """

    B_raw = np.zeros(
        (
            N_FEATURES,
            N
        ),
        dtype=float
    )


    exact_supports = set()


    for output_index in range(N):

        polynomial = sp.Poly(
            sp.expand(
                field_symbolic[
                    output_index
                ]
            ),
            *x_symbols
        )


        for exponent, coefficient in polynomial.terms():

            if coefficient == 0:
                continue


            total_degree = sum(
                exponent
            )


            # Constant terms should not occur.
            if total_degree == 0:

                raise RuntimeError(
                    "Unexpected constant term "
                    f"in output {output_index + 1}"
                )


            if exponent not in exponent_to_feature:

                raise RuntimeError(
                    "Oracle term lies outside "
                    "learner dictionary: "
                    f"output={output_index + 1}, "
                    f"exponent={exponent}, "
                    f"coefficient={coefficient}"
                )


            feature_index = (
                exponent_to_feature[
                    exponent
                ]
            )


            B_raw[
                feature_index,
                output_index
            ] = float(
                coefficient
            )


            variable_support = {
                j + 1

                for j, power in enumerate(
                    exponent
                )

                if power > 0
            }


            structural_support = tuple(
                sorted(
                    variable_support
                    |
                    {
                        output_index + 1
                    }
                )
            )


            exact_supports.add(
                structural_support
            )


    return (
        B_raw,
        tuple(
            sorted(
                exact_supports,
                key=lambda s: (
                    len(s),
                    s
                )
            )
        )
    )


(
    B0_oracle_raw,
    oracle_supports_F0,
) = symbolic_field_to_raw_coefficients(
    F0_oracle_symbolic
)


(
    B1_oracle_raw,
    oracle_supports_F1,
) = symbolic_field_to_raw_coefficients(
    F1_oracle_symbolic
)


# --------------------------------------------------------------
# 5. Oracle support summaries
# --------------------------------------------------------------

oracle_count_by_size_F0 = Counter(
    len(support)
    for support in oracle_supports_F0
)


oracle_count_by_size_F1 = Counter(
    len(support)
    for support in oracle_supports_F1
)


# --------------------------------------------------------------
# 6. Structural comparison
# --------------------------------------------------------------

def compare_support_sets(
    learned_supports,
    oracle_supports,
):

    learned_set = set(
        learned_supports
    )

    oracle_set = set(
        oracle_supports
    )


    true_positive = tuple(
        sorted(
            learned_set
            &
            oracle_set,

            key=lambda s: (
                len(s),
                s
            )
        )
    )


    false_positive = tuple(
        sorted(
            learned_set
            -
            oracle_set,

            key=lambda s: (
                len(s),
                s
            )
        )
    )


    false_negative = tuple(
        sorted(
            oracle_set
            -
            learned_set,

            key=lambda s: (
                len(s),
                s
            )
        )
    )


    precision = (
        len(true_positive)
        /
        len(learned_set)
        if learned_set
        else 1.0
    )


    recall = (
        len(true_positive)
        /
        len(oracle_set)
        if oracle_set
        else 1.0
    )


    return {
        "TP":
            true_positive,

        "FP":
            false_positive,

        "FN":
            false_negative,

        "precision":
            precision,

        "recall":
            recall,
    }


comparison_F0 = compare_support_sets(
    selected_supports_F0,
    oracle_supports_F0,
)


comparison_F1 = compare_support_sets(
    selected_supports_F1_ext,
    oracle_supports_F1,
)


# --------------------------------------------------------------
# 7. Support diagnostics by interaction order
# --------------------------------------------------------------

def error_count_by_size(
    supports
):
    return dict(
        sorted(
            Counter(
                len(s)
                for s in supports
            ).items()
        )
    )


# --------------------------------------------------------------
# 8. Oracle field values on train/test states
# --------------------------------------------------------------

F0_oracle_train = (
    Theta_train_raw
    @ B0_oracle_raw
)

F0_oracle_test = (
    Theta_test_raw
    @ B0_oracle_raw
)


F1_oracle_train = (
    Theta_train_raw
    @ B1_oracle_raw
)

F1_oracle_test = (
    Theta_test_raw
    @ B1_oracle_raw
)


# --------------------------------------------------------------
# 9. Convert learned F1 coefficients to raw basis
# --------------------------------------------------------------

B1_hat_raw_ext = (
    B1_hat_scaled_ext
    /
    feature_scale[:, None]
)


# --------------------------------------------------------------
# 10. Generator errors
#
# Separate:
#
#   A_data       vs oracle F0
#   learned F0   vs oracle F0
#
#   F1_target    vs oracle F1
#   learned F1   vs oracle F1
#
# This tells us where any residual error enters.
# --------------------------------------------------------------

A_target_error_train = (
    relative_field_error(
        A_train_data,
        F0_oracle_train,
    )
)

A_target_error_test = (
    relative_field_error(
        A_test_data,
        F0_oracle_test,
    )
)


F0_learned_error_train = (
    relative_field_error(
        A_train_hat,
        F0_oracle_train,
    )
)

F0_learned_error_test = (
    relative_field_error(
        A_test_hat,
        F0_oracle_test,
    )
)


F1_target_error_train = (
    relative_field_error(
        F1_train_target,
        F1_oracle_train,
    )
)

F1_target_error_test = (
    relative_field_error(
        F1_test_target,
        F1_oracle_test,
    )
)


F1_learned_error_train = (
    relative_field_error(
        F1_train_hat_ext,
        F1_oracle_train,
    )
)

F1_learned_error_test = (
    relative_field_error(
        F1_test_hat_ext,
        F1_oracle_test,
    )
)


# --------------------------------------------------------------
# 11. Coefficient-space errors
# --------------------------------------------------------------

F0_coefficient_relerr = float(
    np.linalg.norm(
        B0_hat_raw
        -
        B0_oracle_raw
    )
    /
    np.linalg.norm(
        B0_oracle_raw
    )
)


F1_coefficient_relerr = float(
    np.linalg.norm(
        B1_hat_raw_ext
        -
        B1_oracle_raw
    )
    /
    np.linalg.norm(
        B1_oracle_raw
    )
)


# --------------------------------------------------------------
# 12. Print oracle validation
# --------------------------------------------------------------

print(
    "================================================"
)

print(
    "ORACLE VALIDATION"
)

print(
    "================================================"
)


print()
print("Oracle structural closure:")

print(
    "  F0 groups =",
    len(
        oracle_supports_F0
    ),
    "| by size =",
    dict(
        sorted(
            oracle_count_by_size_F0.items()
        )
    )
)

print(
    "  F1 groups =",
    len(
        oracle_supports_F1
    ),
    "| by size =",
    dict(
        sorted(
            oracle_count_by_size_F1.items()
        )
    )
)


print()
print("F0 structural recovery:")

print(
    "  learned =",
    len(
        selected_supports_F0
    )
)

print(
    "  oracle  =",
    len(
        oracle_supports_F0
    )
)

print(
    "  TP / FP / FN =",
    len(
        comparison_F0["TP"]
    ),
    "/",
    len(
        comparison_F0["FP"]
    ),
    "/",
    len(
        comparison_F0["FN"]
    )
)

print(
    "  precision =",
    f"{comparison_F0['precision']:.6f}"
)

print(
    "  recall    =",
    f"{comparison_F0['recall']:.6f}"
)

print(
    "  FP by size =",
    error_count_by_size(
        comparison_F0["FP"]
    )
)

print(
    "  FN by size =",
    error_count_by_size(
        comparison_F0["FN"]
    )
)


print()
print("F1 structural recovery:")

print(
    "  learned =",
    len(
        selected_supports_F1_ext
    )
)

print(
    "  oracle  =",
    len(
        oracle_supports_F1
    )
)

print(
    "  TP / FP / FN =",
    len(
        comparison_F1["TP"]
    ),
    "/",
    len(
        comparison_F1["FP"]
    ),
    "/",
    len(
        comparison_F1["FN"]
    )
)

print(
    "  precision =",
    f"{comparison_F1['precision']:.6f}"
)

print(
    "  recall    =",
    f"{comparison_F1['recall']:.6f}"
)

print(
    "  FP by size =",
    error_count_by_size(
        comparison_F1["FP"]
    )
)

print(
    "  FN by size =",
    error_count_by_size(
        comparison_F1["FN"]
    )
)


print()
print("False positives F1:")

for support in comparison_F1[
    "FP"
]:
    print(
        " ",
        support
    )


print()
print("False negatives F1:")

for support in comparison_F1[
    "FN"
]:
    print(
        " ",
        support
    )


print()
print("Resolution / correction target errors:")

print(
    "  A_data vs oracle F0 train =",
    f"{A_target_error_train:.3e}"
)

print(
    "  A_data vs oracle F0 test  =",
    f"{A_target_error_test:.3e}"
)

print(
    "  F1_target vs oracle F1 train =",
    f"{F1_target_error_train:.3e}"
)

print(
    "  F1_target vs oracle F1 test  =",
    f"{F1_target_error_test:.3e}"
)


print()
print("Final learned generator errors:")

print(
    "  learned F0 vs oracle train =",
    f"{F0_learned_error_train:.3e}"
)

print(
    "  learned F0 vs oracle test  =",
    f"{F0_learned_error_test:.3e}"
)

print(
    "  learned F1 vs oracle train =",
    f"{F1_learned_error_train:.3e}"
)

print(
    "  learned F1 vs oracle test  =",
    f"{F1_learned_error_test:.3e}"
)


print()
print("Coefficient-space errors:")

print(
    "  F0 coefficient relative error =",
    f"{F0_coefficient_relerr:.3e}"
)

print(
    "  F1 coefficient relative error =",
    f"{F1_coefficient_relerr:.3e}"
)

ORACLE VALIDATION

Oracle structural closure:
  F0 groups = 25 | by size = {1: 8, 2: 15, 3: 2}
  F1 groups = 58 | by size = {1: 3, 2: 25, 3: 25, 4: 5}

F0 structural recovery:
  learned = 25
  oracle  = 25
  TP / FP / FN = 25 / 0 / 0
  precision = 1.000000
  recall    = 1.000000
  FP by size = {}
  FN by size = {}

F1 structural recovery:
  learned = 59
  oracle  = 58
  TP / FP / FN = 58 / 1 / 0
  precision = 0.983051
  recall    = 1.000000
  FP by size = {1: 1}
  FN by size = {}

False positives F1:
  (5,)

False negatives F1:

Resolution / correction target errors:
  A_data vs oracle F0 train = 2.037e-09
  A_data vs oracle F0 test  = 2.087e-09
  F1_target vs oracle F1 train = 2.087e-05
  F1_target vs oracle F1 test  = 2.072e-05

Final learned generator errors:
  learned F0 vs oracle train = 1.894e-09
  learned F0 vs oracle test  = 1.940e-09
  learned F1 vs oracle train = 2.035e-05
  learned F1 vs oracle test  = 2.022e-05

Coefficient-space errors:
  F0 coefficient relative error = 9.

In [1]:
from TSC_AGLASSO import TSCInference

In [17]:
model = TSCInference(
    max_interaction_order=4,
    max_polynomial_degree=3,
    temporal_order=1,
)

result = model.fit(
    X0=learner_data["X0_train"],
    XF=learner_data["forward_train"],
    eps=learner_data["epsilon_values"],

    X0_test=learner_data["X0_test"],
    XF_test=learner_data["forward_test"],
)

TSC configuration:
  max_interaction_order = 4
  max_polynomial_degree = 3
  temporal_order = 1
  extrapolation eps = [0.005  0.0075 0.01   0.015 ]
  library features/output = 164
  structural groups = 162

Starting Adaptive Group LASSO path for F^(0) (80 lambdas)...
  [001/80] lambda=1.315e-01 | groups=  0 | iter=   3 | elapsed=0.0s
  [006/80] lambda=5.486e-02 | groups=  6 | iter=  71 | elapsed=1.5s
  [011/80] lambda=2.288e-02 | groups=  8 | iter= 184 | elapsed=4.2s
  [016/80] lambda=9.545e-03 | groups= 12 | iter= 388 | elapsed=10.6s
  [021/80] lambda=3.981e-03 | groups= 20 | iter= 638 | elapsed=21.9s
  [026/80] lambda=1.661e-03 | groups= 20 | iter= 810 | elapsed=37.6s
  [031/80] lambda=6.927e-04 | groups= 20 | iter= 879 | elapsed=56.8s
  [036/80] lambda=2.889e-04 | groups= 20 | iter= 878 | elapsed=75.8s
  [041/80] lambda=1.205e-04 | groups= 20 | iter= 843 | elapsed=92.3s
  [046/80] lambda=5.027e-05 | groups= 20 | iter= 798 | elapsed=105.8s
  [051/80] lambda=2.097e-05 | groups= 20 | i

In [8]:
model = TSCInference(
    max_interaction_order=4,
    max_polynomial_degree=3,
    temporal_order=1,
)

result = model.fit(
    X0=learner_data["X0_train"],
    XF=learner_data["forward_train"],
    eps=learner_data["epsilon_values"],

    X0_test=learner_data["X0_test"],
    XF_test=learner_data["forward_test"],
)

TSC configuration:
  max_interaction_order = 4
  max_polynomial_degree = 3
  temporal_order = 1
  extrapolation eps = [0.005  0.0075 0.01   0.015 ]
  library features/output = 164
  structural groups = 162

Starting Adaptive Group LASSO path for F^(0) (80 lambdas)...
  [001/80] lambda=1.315e-01 | groups=  0 | iter=   3 | elapsed=0.0s
  [006/80] lambda=5.486e-02 | groups=  6 | iter=  71 | elapsed=1.7s
  [011/80] lambda=2.288e-02 | groups=  8 | iter= 184 | elapsed=6.2s
  [016/80] lambda=9.545e-03 | groups= 12 | iter= 388 | elapsed=12.8s
  [021/80] lambda=3.981e-03 | groups= 20 | iter= 638 | elapsed=22.3s
  [026/80] lambda=1.661e-03 | groups= 20 | iter= 810 | elapsed=38.1s
  [031/80] lambda=6.927e-04 | groups= 20 | iter= 879 | elapsed=59.3s
  [036/80] lambda=2.889e-04 | groups= 20 | iter= 878 | elapsed=88.3s
  [041/80] lambda=1.205e-04 | groups= 20 | iter= 843 | elapsed=110.2s
  [046/80] lambda=5.027e-05 | groups= 20 | iter= 798 | elapsed=129.4s
  [051/80] lambda=2.097e-05 | groups= 20 | 

In [9]:
display(result.structure_table())

,order,support,F^(0),F^(1),origin
0,1,"(1,)",True,True,mixed
1,1,"(2,)",True,True,mixed
2,1,"(3,)",True,True,mixed
3,1,"(4,)",True,False,persistent-only
4,1,"(5,)",True,True,mixed
...,...,...,...,...,...
58,4,"(1, 2, 3, 4)",False,True,temporal-only
59,4,"(1, 2, 5, 8)",False,True,temporal-only
60,4,"(2, 4, 5, 8)",False,True,temporal-only
61,4,"(2, 5, 6, 8)",False,True,temporal-only


In [12]:
table = result.structure_table()

display(
    table[
        table["support"].isin([
            (1, 2, 3),
            (2, 5, 8),
            (6, 8),
            (1, 2, 5, 8),
        ])
    ]
)

,order,support,F^(0),F^(1),origin
31,2,"(6, 8)",False,True,temporal-only
33,3,"(1, 2, 3)",True,True,mixed
47,3,"(2, 5, 8)",True,True,mixed
59,4,"(1, 2, 5, 8)",False,True,temporal-only


In [13]:
beta0 = result.beta_S((1, 2, 3), temporal_order=0)
beta1 = result.beta_S((1, 2, 3), temporal_order=1)

print(beta0)
print(beta1)

[ 2.00000018e-02  2.00000006e-02  2.00000014e-02  1.62056691e-09
  7.09601869e-10  8.61103816e-10 -6.23271967e-10  2.55296578e-10
  2.32765873e-09  7.90195368e-10  1.72671184e-09  1.78455935e-09]
[ 5.30052755e-02 -5.34681303e-02  1.61874108e-03  1.24538915e-02
 -1.26914752e-02  1.43327532e-02  1.78815406e-07 -1.11343694e-03
 -1.06352961e-06 -1.37659157e-02  1.32582908e-02 -1.24446882e-02]


In [14]:
display(
    result.sector_coefficients(
        (1, 2, 3),
        temporal_order=0,
        nonzero_only=True
    )
)

,output,monomial,coefficient
0,3,x1*x2,2.000000e-02
1,2,x1*x3,2.000000e-02
2,1,x2*x3,2.000000e-02
3,3,x1^2*x2,1.620567e-09
4,2,x1^2*x3,7.096019e-10
5,3,x1*x2^2,8.611038e-10
6,1,x1*x2*x3,-6.232720e-10
7,2,x1*x2*x3,2.552966e-10
8,3,x1*x2*x3,2.327659e-09
9,2,x1*x3^2,7.901954e-10


In [20]:
from TSC_AGLASSO_v3 import TSCInference

In [21]:
model = TSCInference(
    max_interaction_order=4,
    max_polynomial_degree=3,
    temporal_order=1,
)

result = model.fit(
    X0=learner_data["X0_train"],
    XF=learner_data["forward_train"],
    eps=learner_data["epsilon_values"],

    X0_test=learner_data["X0_test"],
    XF_test=learner_data["forward_test"],
)

TSC configuration:
  max_interaction_order = 4
  max_polynomial_degree = 3
  temporal_order = 1
  extrapolation eps = [0.005  0.0075 0.01   0.015 ]
  library features/output = 164
  structural groups = 162

Starting KKT-certified working-set Adaptive Group LASSO for F^(0) (80 lambdas)...
  pilot = adaptive Ridge | alpha range = [6.827e-12, 6.827e-12]
  [001/80] lambda=1.315e-01 | groups=  0 | WS=  0 | KKT+=  0 | iter=   0 | elapsed=0.0s
  [006/80] lambda=5.486e-02 | groups=  6 | WS=  6 | KKT+=  1 | iter= 116 | elapsed=0.3s
  [011/80] lambda=2.288e-02 | groups=  8 | WS=  8 | KKT+=  1 | iter= 226 | elapsed=1.0s
  [016/80] lambda=9.545e-03 | groups= 12 | WS= 12 | KKT+=  3 | iter= 675 | elapsed=3.7s
  [021/80] lambda=3.981e-03 | groups= 20 | WS= 20 | KKT+=  0 | iter= 634 | elapsed=30.3s
  [026/80] lambda=1.661e-03 | groups= 20 | WS= 20 | KKT+=  0 | iter= 806 | elapsed=63.5s
  [031/80] lambda=6.927e-04 | groups= 20 | WS= 20 | KKT+=  0 | iter= 873 | elapsed=100.8s
  [036/80] lambda=2.889e-04

In [22]:
# ================================================================
# Regression audit — exact symbolic oracle vs inferred supports
# ================================================================

from collections import Counter
import sympy as sp

# --------------------------------------------------------------
# 1. Exact symbolic F^(0)
#
# Six equal-duration snapshots:
#
#     F^(0) = (1/6) sum_r G_r
# --------------------------------------------------------------

F0_oracle_symbolic = sum(
    snapshot_fields_symbolic,
    sp.zeros(N, 1)
) / len(snapshot_fields_symbolic)

F0_oracle_symbolic = sp.Matrix([
    sp.expand(expr)
    for expr in F0_oracle_symbolic
])


# --------------------------------------------------------------
# 2. Exact symbolic F^(1)
#
# Chronological protocol:
#
#     G1 -> G2 -> ... -> G6
#
# with each snapshot duration epsilon / 6.
#
# For the effective generator:
#
#     F_eff = F^(0) + epsilon F^(1) + O(epsilon^2)
#
# the BCH first-order correction is
#
#     F^(1) = (1 / (2 * 6^2)) sum_{s>r} [G_s, G_r]
#           = (1 / 72) sum_{s>r} [G_s, G_r]
#
# Overall Lie-bracket sign does not affect support recovery.
# --------------------------------------------------------------

def lie_bracket(F, G):
    """
    Vector-field Lie bracket.

        [F, G] = J_G F - J_F G

    Swapping the convention only flips the global sign of F^(1),
    so the structural support oracle is unchanged.
    """
    JF = F.jacobian(x_symbols)
    JG = G.jacobian(x_symbols)

    return sp.Matrix([
        sp.expand(expr)
        for expr in (JG * F - JF * G)
    ])


F1_oracle_symbolic = sp.zeros(N, 1)

for s in range(len(snapshot_fields_symbolic)):
    for r in range(s):
        F1_oracle_symbolic += lie_bracket(
            snapshot_fields_symbolic[r],
            snapshot_fields_symbolic[s],
        )

F1_oracle_symbolic /= (
    2 * len(snapshot_fields_symbolic)**2
)

F1_oracle_symbolic = sp.Matrix([
    sp.expand(expr)
    for expr in F1_oracle_symbolic
])


# --------------------------------------------------------------
# 3. Extract minimal structural supports
#
# For a monomial appearing in output i:
#
#     support = {output i} U {variables appearing in monomial}
#
# Node labels are returned 1-based, exactly matching TSC_AGLASSO.
# --------------------------------------------------------------

def symbolic_structural_supports(field):
    supports = set()

    for output_index, expr in enumerate(field):

        poly = sp.Poly(
            sp.expand(expr),
            *x_symbols
        )

        for powers, coefficient in poly.terms():

            # Exact symbolic zero check
            if sp.simplify(coefficient) == 0:
                continue

            variables = {
                j + 1
                for j, power in enumerate(powers)
                if power > 0
            }

            support = tuple(sorted(
                variables | {output_index + 1}
            ))

            supports.add(support)

    return tuple(sorted(
        supports,
        key=lambda s: (len(s), s)
    ))


oracle_F0 = symbolic_structural_supports(
    F0_oracle_symbolic
)

oracle_F1 = symbolic_structural_supports(
    F1_oracle_symbolic
)


# --------------------------------------------------------------
# 4. Inferred supports
# --------------------------------------------------------------

inferred_F0 = tuple(
    sorted(
        result.F0.selected_supports,
        key=lambda s: (len(s), s)
    )
)

inferred_F1 = tuple(
    sorted(
        result.F1.selected_supports,
        key=lambda s: (len(s), s)
    )
)


# --------------------------------------------------------------
# 5. Comparison utility
# --------------------------------------------------------------

def compare_supports(
    oracle,
    inferred,
    label,
):
    oracle_set = set(oracle)
    inferred_set = set(inferred)

    TP = oracle_set & inferred_set
    FP = inferred_set - oracle_set
    FN = oracle_set - inferred_set

    precision = (
        len(TP) / len(inferred_set)
        if inferred_set
        else 1.0
    )

    recall = (
        len(TP) / len(oracle_set)
        if oracle_set
        else 1.0
    )

    f1_score = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    print("=" * 64)
    print(label)
    print("=" * 64)

    print(
        "Oracle groups by support size =",
        dict(sorted(
            Counter(
                len(s)
                for s in oracle_set
            ).items()
        ))
    )

    print(
        "Inferred groups by support size =",
        dict(sorted(
            Counter(
                len(s)
                for s in inferred_set
            ).items()
        ))
    )

    print()

    print("Oracle supports   =", len(oracle_set))
    print("Inferred supports =", len(inferred_set))

    print()

    print("TP =", len(TP))
    print("FP =", len(FP))
    print("FN =", len(FN))

    print()

    print(f"Precision = {precision:.6f}")
    print(f"Recall    = {recall:.6f}")
    print(f"F1 score  = {f1_score:.6f}")

    print()

    print(
        "Exact support-set match =",
        oracle_set == inferred_set
    )

    if FP:
        print()
        print("False positives:")
        for support in sorted(
            FP,
            key=lambda s: (len(s), s)
        ):
            print(" ", support)

    if FN:
        print()
        print("False negatives:")
        for support in sorted(
            FN,
            key=lambda s: (len(s), s)
        ):
            print(" ", support)

    return {
        "oracle": oracle_set,
        "inferred": inferred_set,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1_score,
    }


# --------------------------------------------------------------
# 6. Run regression audit
# --------------------------------------------------------------

audit_F0 = compare_supports(
    oracle_F0,
    inferred_F0,
    "F^(0) structural recovery"
)

print()

audit_F1 = compare_supports(
    oracle_F1,
    inferred_F1,
    "F^(1) structural recovery"
)


# --------------------------------------------------------------
# 7. Frozen N=8 regression checks
# --------------------------------------------------------------

assert len(oracle_F0) == 25

assert Counter(
    len(s)
    for s in oracle_F0
) == {
    1: 8,
    2: 15,
    3: 2,
}

assert len(oracle_F1) == 58

assert len(audit_F0["FN"]) == 0
assert len(audit_F0["FP"]) == 0

assert len(audit_F1["FN"]) == 0

print()
print("=" * 64)
print("Frozen N=8 regression summary")
print("=" * 64)

print(
    "F^(0):",
    f"{len(audit_F0['TP'])}/{len(oracle_F0)} oracle supports recovered",
)

print(
    "F^(1):",
    f"{len(audit_F1['TP'])}/{len(oracle_F1)} oracle supports recovered",
)

print(
    "F^(1) false positives =",
    sorted(
        audit_F1["FP"],
        key=lambda s: (len(s), s)
    )
)

print(
    "k_star =",
    result.k_star
)

print(
    "F0 KKT certified =",
    result.F0.kkt_certified
)

print(
    "F1 KKT certified =",
    result.F1.kkt_certified
)

print(
    "F0 total KKT activations =",
    result.F0.total_kkt_reactivations
)

print(
    "F1 total KKT activations =",
    result.F1.total_kkt_reactivations
)

F^(0) structural recovery
Oracle groups by support size = {1: 8, 2: 15, 3: 2}
Inferred groups by support size = {1: 8, 2: 15, 3: 2}

Oracle supports   = 25
Inferred supports = 25

TP = 25
FP = 0
FN = 0

Precision = 1.000000
Recall    = 1.000000
F1 score  = 1.000000

Exact support-set match = True

F^(1) structural recovery
Oracle groups by support size = {1: 3, 2: 25, 3: 25, 4: 5}
Inferred groups by support size = {1: 4, 2: 25, 3: 25, 4: 5}

Oracle supports   = 58
Inferred supports = 59

TP = 58
FP = 1
FN = 0

Precision = 0.983051
Recall    = 1.000000
F1 score  = 0.991453

Exact support-set match = False

False positives:
  (5,)

Frozen N=8 regression summary
F^(0): 25/25 oracle supports recovered
F^(1): 58/58 oracle supports recovered
F^(1) false positives = [(5,)]
k_star = 4
F0 KKT certified = True
F1 KKT certified = True
F0 total KKT activations = 25
F1 total KKT activations = 59


In [23]:
table = result.structure_table()

display(
    table[
        table["support"].isin([
            (1, 2, 3),
            (2, 5, 8),
            (6, 8),
            (1, 2, 5, 8),
        ])
    ]
)

,order,support,F^(0),F^(1),origin
31,2,"(6, 8)",False,True,temporal-only
33,3,"(1, 2, 3)",True,True,mixed
47,3,"(2, 5, 8)",True,True,mixed
59,4,"(1, 2, 5, 8)",False,True,temporal-only


In [24]:
beta0 = result.beta_S((1, 2, 3), temporal_order=0)
beta1 = result.beta_S((1, 2, 3), temporal_order=1)

print(beta0)
print(beta1)

[ 2.00000018e-02  2.00000006e-02  2.00000014e-02  1.62056691e-09
  7.09601869e-10  8.61103816e-10 -6.23271967e-10  2.55296578e-10
  2.32765873e-09  7.90195368e-10  1.72671184e-09  1.78455935e-09]
[ 5.30052755e-02 -5.34681303e-02  1.61874108e-03  1.24538915e-02
 -1.26914752e-02  1.43327532e-02  1.78815406e-07 -1.11343694e-03
 -1.06352961e-06 -1.37659157e-02  1.32582908e-02 -1.24446882e-02]


In [25]:
display(
    result.sector_coefficients(
        (1, 2, 3),
        temporal_order=0,
        nonzero_only=True
    )
)

,output,monomial,coefficient
0,3,x1*x2,2.000000e-02
1,2,x1*x3,2.000000e-02
2,1,x2*x3,2.000000e-02
3,3,x1^2*x2,1.620567e-09
4,2,x1^2*x3,7.096019e-10
5,3,x1*x2^2,8.611038e-10
6,1,x1*x2*x3,-6.232720e-10
7,2,x1*x2*x3,2.552966e-10
8,3,x1*x2*x3,2.327659e-09
9,2,x1*x3^2,7.901954e-10
